In [ ]:
!pip install PyAudio

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import scipy.fftpack as fourier
import time # Importamos time para hacer una pausa corta

# Configuración de Matplotlib
matplotlib.use('TkAgg')

# --- 1. Definición de la Señal Base ---
frec = 2                                           # Frecuencia de la señal (2 Hz)
Fs = 100                                   # Frecuencia de muestreo (100 Hz)


t = np.arange(0, duracion, 1/Fs)           # Eje del tiempo
y_senoidal_base = np.sin(2 * np.pi * frec * t)     # Señal Senoidal pura

# --- 2. Preparación del Gráfico ---
# Definir el Eje de Frecuencias (x_fft) una sola vez
x_fft = fourier.fftfreq(Fs, 1.0/f_muestreo)
mitad = Fs // 2 + 1
x_fft_plot = x_fft[:mitad]

fig, (ax1) = plt.subplots(1)
plt.title(f'Espectro de Frecuencias (Senoidal de {frec} Hz + Ruido)')

# Inicializamos el gráfico (con ceros o la FFT de la señal base)
M_gk_inicial = np.abs(fourier.fft(y_senoidal_base)) / Fs
line_fft, = ax1.plot(x_fft_plot, M_gk_inicial[:mitad], 'b')

# Ajustar límites y etiquetas
ax1.set_xlim(0, Fs / 2)
ax1.set_ylim(0, np.max(M_gk_inicial) * 2) # Damos margen para el ruido
ax1.set_xlabel('Frecuencia (Hz)')
ax1.set_ylabel('Magnitud Normalizada')
ax1.grid(True)

fig.show()

# --- 3. Bucle para Añadir Ruido y Actualizar la FFT ---
# Parámetro que controla la intensidad del ruido (ajustar para cambiar el efecto)
AMPLITUD_RUIDO = 0.5 

print("Iniciando visualización dinámica con ruido...")
try:
    while True:
        # 3.1. Generar Ruido Blanco
        # np.random.normal(media, desviación_estándar, tamaño)
        # Esto genera 100 puntos de ruido aleatorio
        ruido = np.random.normal(0, AMPLITUD_RUIDO, Fs)
        
        # 3.2. Sumar el Ruido a la Señal Base
        y_senoidal_ruidosa = y_senoidal_base + ruido
        
        # 3.3. Calcular la FFT de la Señal Ruidosa
        M_gk_total = np.abs(fourier.fft(y_senoidal_ruidosa))
        # Normalizamos y tomamos solo la mitad positiva
        M_gk_plot = M_gk_total[:mitad] / Fs
        
        # 3.4. Actualizar la Gráfica
        line_fft.set_ydata(M_gk_plot)
               
        # Redibujar la gráfica
        fig.canvas.draw()
        fig.canvas.flush_events()
        
        # Añadir una pequeña pausa para que la CPU respire (opcional)
        time.sleep(0.05) 
        
except Exception as e:
    print(f"La ventana de gráfica fue cerrada o ocurrió un error: {e}")

finally:
    plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import pyaudio as pa 
import numpy as np
import struct 
import scipy.fftpack as fourier


matplotlib.use('TkAgg')
#%matplotlib notebook

FRAMES = 1024*8                                   # Tamaño del paquete a procesar
FORMAT = pa.paInt16                               # Formato de lectura INT 16 bits
CHANNELS = 1
Fs = 44100                                        # Frecuencia de muestreo típica para audio

p = pa.PyAudio()

stream = p.open(                                  # Abrimos el canal de audio con los parámeteros de configuración
    format = FORMAT,
    channels = CHANNELS,
    rate = Fs,
    input=True,
    output=True,
    frames_per_buffer=FRAMES
)

## Creamos una gráfica con 2 subplots y configuramos los ejes

frec = 2
f_muestreo = 100
t = np.arange(0, 1, 1/f_muestreo)
y_fft = np.sin(2 * np.pi * frec * t)
plt.title('Señal Senoidal (NumPy)')
plt.plot(t, y_fft)




fig, (ax1) = plt.subplots(1)




line_fft, = ax1.semilogx(x_fft, np.random.rand(FRAMES), 'b')

fig.show()


while True:
    M_gk = abs(fourier.fft(y_fft)/1)            # Calculamos la FFT y la Magnitud de la FFT del paqute de datos

    ax1.set_ylim(0,np.max(M_gk+10)) 
    line_fft.set_ydata(M_gk)                           # Asigmanos la Magnitud de la FFT a la curva del espectro 
    
    fig.canvas.draw()
    fig.canvas.flush_events()

In [ ]:
import matplotlib.pyplot as plt
import pyaudio as pa 
import numpy as np
import struct 
import scipy.fftpack as fourier


matplotlib.use('TkAgg')
#%matplotlib notebook

FRAMES = 1024*8                                   # Tamaño del paquete a procesar
FORMAT = pa.paInt16                               # Formato de lectura INT 16 bits
CHANNELS = 1
Fs = 44100                                        # Frecuencia de muestreo típica para audio

p = pa.PyAudio()

stream = p.open(                                  # Abrimos el canal de audio con los parámeteros de configuración
    format = FORMAT,
    channels = CHANNELS,
    rate = Fs,
    input=True,
    output=True,
    frames_per_buffer=FRAMES
)

## Creamos una gráfica con 2 subplots y configuramos los ejes

fig, (ax,ax1) = plt.subplots(2)

x_audio = np.arange(0,FRAMES,1)
x_fft = np.linspace(0, Fs, FRAMES)

line, = ax.plot(x_audio, np.random.rand(FRAMES),'r')
line_fft, = ax1.semilogx(x_fft, np.random.rand(FRAMES), 'b')

ax.set_ylim(-2500,2500)
ax.ser_xlim = (0,FRAMES)

Fmin = 1
Fmax = 5000
ax1.set_xlim(Fmin,Fmax)

fig.show()


while True:
    data = stream.read(FRAMES)                         # Leemos paquetes de longitud FRAMES
    dataInt = struct.unpack(str(FRAMES) + 'h', data)   # Convertimos los datos que se encuentran empaquetados en bytes
    
    line.set_ydata(dataInt)                            # Asignamos los datos a la curva de la variación temporal
    
    M_gk = abs(fourier.fft(dataInt)/FRAMES)            # Calculamos la FFT y la Magnitud de la FFT del paqute de datos

    ax1.set_ylim(0,np.max(M_gk+10)) 
    line_fft.set_ydata(M_gk)                           # Asigmanos la Magnitud de la FFT a la curva del espectro 
    
    fig.canvas.draw()
    fig.canvas.flush_events()

In [ ]:
!pip install Keyboard

In [ ]:
# ==============================
# Importación de librerías
# ==============================
import cv2              # Librería OpenCV para visión por computadora
import mediapipe as mp  # Librería MediaPipe para detección de manos
import numpy as np      # Librería NumPy para cálculos numéricos
import pyautogui        # Librería para controlar el mouse y teclado
import time
import keyboard
import scipy.signal as sig
import math
# ==============================
# Configuración de MediaPipe y cámara
# ==============================
mp_hands = mp.solutions.hands            # Modelo de detección de manos de MediaPipe
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) # Abre la cámara (índice 0), CAP_DSHOW es para Windows

window_size = 1000  # Cuántos frames promediar
positions = []   # Lista circular de posiciones previas

Fs = 100000
fc = 4000  # Frecuencia de corte del filtro
orden = 1  # Orden del filtro
b, a = sig.butter(orden, fc/(0.5*Fs), btype='low', analog=False, output='ba')



def suavizar_pos(x, y):
    positions.append((x, y))
    if len(positions) > window_size:
        positions.pop(0)


    #descomprime la posicion almacenada(x,y) en 2 variables
    xs, ys = zip(*positions)

    #FIltra la secuencia entera en la ventana
    x0_filtered = sig.lfilter(b, a, xs)
    y0_filtered = sig.lfilter(b, a, ys)

    #toma el ultimo elemento
    x_suavizado = x0_filtered[-1]
    y_suavizado = y0_filtered[-1]

    return x_suavizado, y_suavizado


# ==============================
# SUB BLOQUES - funciones principales
# ==============================
def mouse_ubicacion(coord_mano):
    mouse_x = mouse_y = 100
    # ==============================
    # Si se detectan manos
    # ==============================
    if coord_mano.multi_hand_landmarks is not None:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            mouse_x = int(hand_landmarks.landmark[0].x * width_frame) #libreria py autogui tamaño pantalla
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)
            mouse_x,mouse_y =    suavizar_pos(mouse_x, mouse_y)

    return mouse_x , mouse_y



def mouse_gesto_operacion(coord_mano):
    BI=BD=0
    if coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            # Ejemplo: dedo índice extendido => click izquierdo
            BD  = hand_landmarks.landmark[8].y > hand_landmarks.landmark[6].y
            BI  = hand_landmarks.landmark[12].y > hand_landmarks.landmark[10].y
    return BD,BI

# ==============================
# BLOQUES - funciones principales
# ==============================
def Deteccion_mano_openCV(frame):
    frame = cv2.flip(frame, 1)      # Voltear la imagen horizontalmente (como espejo)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) #lo necesita?
    coord_mano = hands.process(frame_rgb) # Procesar frame con MediaPipe para detectar manos
    return coord_mano

def Deteccion_gestos(coord_mano):
    mouse_x , mouse_y = mouse_ubicacion(coord_mano)
    BD, BI = mouse_gesto_operacion(coord_mano)

    return BD , BI , mouse_x , mouse_y


def f_tecla(FF,boton,tecla):
# ===== Control botón derecho (usa FFD) =====
    if FF == 0 and boton == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1                       # Modifica el estado global FFD

    if FF == 1 and boton == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0                       # Modifica el estado global FFD
    # ===== Fin control botón derecho =====
    return FF

FFD = FFI = 0
def mouse_virtual(x_frame, y_frame , boton_izq, boton_der):
    global FFD,FFI
    width_pantalla,height_pantalla = pyautogui.size()

    x = (int)((x_frame - 1/2*width_frame)*2.2*width_pantalla/width_frame)
    y = (int)((y_frame - 1/2*height_frame)*2.2*height_pantalla/height_frame)

    # ... (Control de movimiento del cursor)
    if (x > 0) and (y > 0):
        try:
            pyautogui.moveTo(x, y)
        except pyautogui.FailSafeException:
            print("Movimiento detenido: mouse llegó a la esquina")
    else:
        pyautogui.moveTo(5, 5)

   # FFD = f_tecla(FFD,boton_der,'right')
    #FFI = f_tecla(FFI,boton_izq,'left')


N = 500
# ==============================
# Bucle principal con MediaPipe Hands
# ==============================
from collections import deque
# 1. Inicializa un array con 1000 ceros: [0, 0, ..., 0]
valores_iniciales = [0] * N

# 2. Crea el deque con la longitud máxima y los valores iniciales.
# Cuando añades un elemento, si ya tiene 1000 elementos, automáticamente
# elimina el primero (el de la izquierda o "más antiguo").
array_fijo = deque(valores_iniciales, maxlen=N)



import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import scipy.fftpack as fourier
import time # Importamos time para hacer una pausa corta

# Configuración de Matplotlib
matplotlib.use('TkAgg')

# --- 1. Definición de la Señal Base ---
Fs = N                                   # Frecuencia de muestreo (100 Hz)



# --- 2. Preparación del Gráfico ---
# Definir el Eje de Frecuencias (x_fft) una sola vez
x_fft = fourier.fftfreq(Fs, 1.0/Fs)
mitad = Fs // 2 + 1
x_fft_plot = x_fft[:mitad]

fig, (ax1) = plt.subplots(1)
plt.title(f'Espectro de Frecuencias (Senoidal de {2} Hz + Ruido)')

y_senoidal_base=0
# Inicializamos el gráfico (con ceros o la FFT de la señal base)
M_gk_inicial = np.abs(fourier.fft(x_fft)) / Fs
line_fft, = ax1.plot(x_fft_plot, M_gk_inicial[:mitad], 'b')


ax1.set_xscale('log')

# Ajustar límites y etiquetas
ax1.set_xlim(0, Fs / 2)
ax1.set_ylim(0, np.max(M_gk_inicial) * 2) # Damos margen para el ruido
ax1.set_xlabel('Frecuencia (Hz)')
ax1.set_ylabel('Magnitud Normalizada')
ax1.grid(True)

fig.show()

from scipy.fftpack import fft, ifft, fftshift
w2 = sig.windows.hann(N)



with mp_hands.Hands(
    static_image_mode=False,        # Procesa video en vivo
    max_num_hands=1,                # Máximo 1 mano detectada
    min_detection_confidence=0.7) as hands:  # Confianza mínima para detección

    while True:
        ret, frame = cap.read()     # Leer un frame de la cámara
        if ret == False:            # Si no hay frame, salir
            break
        ax1.set_ylim(0, 0.5) # Damos margen para el ruido

        
        height_frame, width_frame, _ = frame.shape   # Obtener dimensiones del frame

        coord_mano = Deteccion_mano_openCV(frame)   # Procesar frame con MediaPipe para detectar manos

        BD, BI,mouse_x , mouse_y = Deteccion_gestos(coord_mano)

        mouse_virtual(mouse_x ,mouse_y , BD, BI)

        x=y=0
        if coord_mano.multi_hand_landmarks:
            for hand_landmarks in coord_mano.multi_hand_landmarks:
                # Ejemplo: dedo índice extendido => click izquierdo
                x  = 100*hand_landmarks.landmark[0].x
                y  = 100*hand_landmarks.landmark[0].y

        hip = math.sqrt( x**2 + y**2 )
        array_fijo.append(hip)

        yw2 = array_fijo*w2 #hanning
        #print(mouse_x)
        #print(mouse_y)
        #print(hip)

        M_gk_total = np.abs(fourier.fft(yw2))
        # Normalizamos y tomamos solo la mitad positiva
        M_gk_plot = M_gk_total[:mitad] / Fs
        
        # 3.4. Actualizar la Gráfica
        line_fft.set_ydata(M_gk_plot)
               
        # Redibujar la gráfica
        fig.canvas.draw()
        fig.canvas.flush_events()

       # print((array_fijo))

        if keyboard.is_pressed("esc"):
            print("Saliste del bucle.")
            break
      #  time.sleep(0.05)
# ==============================
# Liberar recursos
# ==============================
cap.release()           # Liberar la cámara
cv2.destroyAllWindows() # Cerrar todas las ventanas de OpenCV
# barra que modifique la frecuencia de corte


In [ ]:
pip install keyboard


In [ ]:
# ==============================
# Importación de librerías
# ==============================
import cv2              # Librería OpenCV para visión por computadora
import mediapipe as mp  # Librería MediaPipe para detección de manos
import numpy as np      # Librería NumPy para cálculos numéricos
import pyautogui        # Librería para controlar el mouse y teclado
import time
import keyboard

# ==============================
# Configuración de MediaPipe y cámara
# ==============================
mp_drawing = mp.solutions.drawing_utils  # Utilidades de dibujo de MediaPipe
mp_hands = mp.solutions.hands            # Modelo de detección de manos de MediaPipe
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) # Abre la cámara (índice 0), CAP_DSHOW es para Windows

# ==============================
# Parámetros de colores y área del juego
# ==============================
color_mouse_pointer = (255, 0, 255)  # Color del puntero virtual (magenta)
# puntero para no ponerle return a la funcion de mouse. Apunta a 0                               NECESARIO
FF = {                  # Diccionario que guarda el estado de los botones
    "btnD": [0],        # "btnD" -> lista con un 0 (simula puntero al botón derecho)
    "btnI": [0]         # "btnI" -> lista con un 0 (simula puntero al botón izquierdo)
}

FF["btnD"][0] = FF["btnI"][0] = 0   # Inicializa ambos botones en estado 0 (no presionado)

# Coordenadas de la "pantalla del juego" dentro del monitor
SCREEN_GAME_X_INI = 150                 # X inicial
SCREEN_GAME_Y_INI = 160                 # Y inicial
SCREEN_GAME_X_FIN = 150 + 780           # X final (150 + ancho del juego)
SCREEN_GAME_Y_FIN = 160 + 450           # Y final (160 + alto del juego)

# Relación de aspecto del área del juego (ancho/alto)
aspect_ratio_screen = (SCREEN_GAME_X_FIN - SCREEN_GAME_X_INI) / (SCREEN_GAME_Y_FIN - SCREEN_GAME_Y_INI)
print("aspect_ratio_screen:", aspect_ratio_screen)

X_Y_INI = 100   # Margen inicial en pixeles para dibujar el área de control en la cámara

# Variables globales para la detección de clicks
prev_BI = 0
last_click_time = 0
hold_start_time = None

# ==============================
# Función para calcular distancia entre dos puntos
# ==============================
def calculate_distance(x1, y1, x2, y2):
    p1 = np.array([x1, y1])     # Punto 1 en un array
    p2 = np.array([x2, y2])     # Punto 2 en un array
    return np.linalg.norm(p1 - p2)  # Devuelve la distancia euclidiana entre ambos puntos




window_size = 10  # Cuántos frames promediar
positions = []   # Lista circular de posiciones previas

def suavizar_pos(x, y):
    positions.append((x, y))
    if len(positions) > window_size:
        positions.pop(0)
    xs, ys = zip(*positions)
    return np.mean(xs), np.mean(ys)



# ==============================
# SUB BLOQUES - funciones principales
# ==============================
def mouse_ubicacion(coord_mano):
    mouse_x = mouse_y = 100
    # ==============================
    # Si se detectan manos
    # ==============================
    l,h = pyautogui.size()
    
    if coord_mano.multi_hand_landmarks is not None:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            mouse_x = int(hand_landmarks.landmark[0].x * l*2-1000) #libreria py autogui tamaño pantalla
            mouse_y = int(hand_landmarks.landmark[0].y * h*2.5-1000)

            mouse_x,mouse_y =    suavizar_pos(mouse_x, mouse_y)
            mouse_x,mouse_y =    suavizar_pos(mouse_x, mouse_y)
            mouse_x,mouse_y =    suavizar_pos(mouse_x, mouse_y)
            mouse_x,mouse_y =    suavizar_pos(mouse_x, mouse_y)
            mouse_x,mouse_y =    suavizar_pos(mouse_x, mouse_y)
            mouse_x,mouse_y =    suavizar_pos(mouse_x, mouse_y)

            cv2.circle(output, ((int)(mouse_x),(int)( mouse_y)), 10, color_mouse_pointer, -1)
    return mouse_x , mouse_y




def mouse_gesto_operacion(coord_mano):
    
    BI=BD=ME=0
 
    if coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            # Ejemplo: dedo índice extendido => click izquierdo
            BD  = hand_landmarks.landmark[8].y > hand_landmarks.landmark[6].y
            BI  = hand_landmarks.landmark[12].y > hand_landmarks.landmark[10].y
                   
    return BD,BI,ME
    
# ==============================
# BLOQUES - funciones principales
# ==============================
def Deteccion_mano_openCV(frame):
    frame = cv2.flip(frame, 1)      # Voltear la imagen horizontalmente (como espejo)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    coord_mano = hands.process(frame_rgb) # Procesar frame con MediaPipe para detectar manos
    return coord_mano

def Deteccion_gestos(coord_mano):

    mouse_x , mouse_y = mouse_ubicacion(coord_mano)
    BD, BI,ME= mouse_gesto_operacion(coord_mano)
    
    if BD == 1 : print(BD)
    if BI == 1 : print(BI)
    return BD , BI , mouse_x , mouse_y


def mouse_virtual(x, y , boton_izq, boton_der, FF):                                         #NECESARIO   
    
    if (x < 5 and y < 5) or (x > 15 and y > 15):
        try:
            pyautogui.moveTo(x, y)          # Mueve el cursor a la posición (x, y)
        except pyautogui.FailSafeException: # Si pyautogui detecta failsafe (mouse en esquina)
            print("Movimiento detenido: mouse llegó a la esquina")  # Muestra mensaje y evita error

   # print(FF["btnI"][0], boton_izq)     # Imprime estado actual del botón izquierdo y el nuevo valor recibido

    # ===== Control botón derecho =====
    if FF["btnD"][0] == 0 and boton_der == 1:   # Si antes estaba suelto y ahora debe presionarse
        pyautogui.mouseDown(button='right')     # Presiona y mantiene botón derecho
        FF["btnD"][0] = 1                       # Actualiza estado a presionado

    if FF["btnD"][0] == 1 and boton_der == 0:   # Si antes estaba presionado y ahora debe soltarse
        pyautogui.mouseUp(button='right')       # Suelta botón derecho
        FF["btnD"][0] = 0                       # Actualiza estado a suelto
    # ===== Fin control botón derecho =====

    # ===== Control botón izquierdo =====
    if FF["btnI"][0] == 0 and boton_izq == 1:   # Si antes estaba suelto y ahora debe presionarse
        pyautogui.mouseDown(button='left')      # Presiona y mantiene botón izquierdo
        FF["btnI"][0] = 1                       # Actualiza estado a presionado

    if FF["btnI"][0] == 1 and boton_izq == 0:   # Si antes estaba presionado y ahora debe soltarse
        pyautogui.mouseUp(button='left')        # Suelta botón izquierdo
        FF["btnI"][0] = 0                       # Actualiza estado a suelto
    # ===== Fin control botón izquierdo =====

# ==============================
# Bucle principal con MediaPipe Hands
# ==============================
with mp_hands.Hands(
    static_image_mode=False,        # Procesa video en vivo
    max_num_hands=1,                # Máximo 1 mano detectada
    min_detection_confidence=0.3) as hands:  # Confianza mínima para detección

    while True:
        ret, frame = cap.read()     # Leer un frame de la cámara
        if ret == False:            # Si no hay frame, salir
            break

        height, width, _ = frame.shape   # Obtener dimensiones del frame

        # ==============================
        # Dibujar área proporcional al área del juego
        # ==============================
        area_width = width - X_Y_INI * 2                  # Ancho del área
        area_height = int(area_width / aspect_ratio_screen)  # Alto según relación de aspecto
        aux_image = np.zeros(frame.shape, np.uint8)       # Imagen auxiliar en negro
        aux_image = cv2.rectangle(aux_image, (X_Y_INI, X_Y_INI),
                                  (X_Y_INI + area_width, X_Y_INI + area_height),
                                  (255, 0, 0), -1)        # Dibujar rectángulo azul
        output = cv2.addWeighted(cv2.flip(frame, 1), 1, aux_image, 0.7, 0) # Combinar cámara y área semitransparente


        coord_mano = Deteccion_mano_openCV(frame)   # Procesar frame con MediaPipe para detectar manos

        BD, BI,mouse_x , mouse_y = Deteccion_gestos(coord_mano)

        mouse_virtual(mouse_x ,mouse_y , BD, BI, FF)


        # ==============================
        # Mostrar resultados
        # ==============================
        cv2.imshow('output', output)  # Mostrar frame con overlay
        if cv2.waitKey(1) & 0xFF == 27:  # Esperar tecla ESC (27) para salir
            break

        if keyboard.is_pressed("esc"):
            print("Saliste del bucle.")
            break

# ==============================
# Liberar recursos
# ==============================
cap.release()           # Liberar la cámara
cv2.destroyAllWindows() # Cerrar todas las ventanas de OpenCV



In [13]:
# agregar al informe
# acomodarlo para qu quede conceptualmeten facil de entender, cambiar de lugar funciones a donde corresponde, asi tmb se puede hacer los dijubos mas faciles
# dibujo general- maqu de estados scroll

# ==============================
# Importación de librerías
# ==============================
import cv2              # Librería OpenCV para visión por computadora
import mediapipe as mp  # Librería MediaPipe para detección de manos
import numpy as np      # Librería NumPy para cálculos numéricos
import pyautogui        # Librería para controlar el mouse y teclado
import time
import keyboard
import scipy.signal as sig
# ==============================
# Configuración de MediaPipe y cámara
# ==============================
mp_hands = mp.solutions.hands            # Modelo de detección de manos de MediaPipe
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) # Abre la cámara (índice 0), CAP_DSHOW es para Windows

window_size = 1000  # Cuántos frames promediar
positions = []   # Lista circular de posiciones previas

Fs = 30
fc = 4  # Frecuencia de corte del filtro
orden = 1  # Orden del filtro
b, a = sig.butter(orden, fc/(0.5*Fs), btype='low', analog=False, output='ba')



def suavizar_pos(x, y):
    positions.append((x, y))
    if len(positions) > window_size:
        positions.pop(0)

    #descomprime la posicion almacenada(x,y) en 2 variables
    xs, ys = zip(*positions)

    #FIltra la secuencia entera en la ventana
    x0_filtered = sig.lfilter(b, a, xs)
    y0_filtered = sig.lfilter(b, a, ys)

    #toma el ultimo elemento
    x_suavizado = x0_filtered[-1]
    y_suavizado = y0_filtered[-1]

    return x_suavizado, y_suavizado


# ==============================
# SUB BLOQUES - funciones principales
# ==============================
estado_scroll = 0
valor_anterior_y = 0
def mouse_gesto_operacion_scroll(coord_mano):
    
    global estado_scroll, valor_anterior_y
    scroll = 0
    
    if coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:   

            gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and estado_scroll == 0
            gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and estado_scroll == 0
            gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

            # ESTADOS
            if(gesto_estado_subir):
                #print("subir")
                estado_scroll = 1

            if(gesto_estado_bajar):
                #print("bajar")
                estado_scroll = 2

            if(gesto_estado_reposo):
                estado_scroll = 0


            # SALIDAS
            #SALIDA SUBIDA
            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                #print("1")
                scroll = 1

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5  and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                #print("0")
                scroll = 0


            #SALIDA BAJADA    
            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
               # print("0")
                scroll = 0

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                #print("-1")
                scroll = -1
            
    else:
        #print("listo")
        estado_scroll = 0     # si no se ve la mano, no la ve la camara

    return scroll,estado_scroll

    
  
    
def mouse_ubicacion(coord_mano):
    mouse_x = mouse_y = 100
    # ==============================
    # Si se detectan manos
    # ==============================
    if coord_mano.multi_hand_landmarks is not None:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            mouse_x = int(hand_landmarks.landmark[0].x * width_frame) #libreria py autogui tamaño pantalla
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)
            mouse_x,mouse_y = suavizar_pos(mouse_x, mouse_y)

    return mouse_x , mouse_y


estadoBD2 = estadoBI2 = 0
def mouse_gesto_operacion(coord_mano):
    BI = (0,0)
    BD = (0,0)
    global estadoBD2,estadoBI2
    if coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            # Ejemplo: dedo índice extendido => click izquierdo
            BD1  = (hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y) and  (hand_landmarks.landmark[5].y < hand_landmarks.landmark[0].y) and (hand_landmarks.landmark[17].y < hand_landmarks.landmark[0].y) 
            BI1  = (hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y) and  (hand_landmarks.landmark[5].y < hand_landmarks.landmark[0].y) and (hand_landmarks.landmark[17].y < hand_landmarks.landmark[0].y)
            BD2 =  ((hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y))
            BI2 =  ((hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y))

            
            if BD2 == 1:
                if estadoBD2==0:
                    #print("1")
                    estadoBD2 = 1
            else:
                estadoBD2 = 0

            if BI2 == 1:
                if estadoBI2==0:
                    #print("2")
                    estadoBI2 = 1
            else:
                estadoBI2 = 0 

            BD = (BD1,BD2) #esta al reves puse la izquierda en la derecha
            BI = (BI1,BI2)
            
    return BD,BI

# ==============================
# BLOQUES - funciones principales
# ==============================
def Deteccion_mano_openCV(frame):
    frame = cv2.flip(frame, 1)      # Voltear la imagen horizontalmente (como espejo)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) #lo necesita?
    coord_mano = hands.process(frame_rgb) # Procesar frame con MediaPipe para detectar manos
    return coord_mano

def Deteccion_gestos(coord_mano):
    mouse_x , mouse_y = mouse_ubicacion(coord_mano)
    BD, BI = mouse_gesto_operacion(coord_mano)

    return BD , BI , mouse_x , mouse_y


def f_tecla(FF,boton,tecla):
# ===== Control botón derecho (usa FFD) =====
    if FF == 0 and boton[0] == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1                       # Modifica el estado global FFD

    if FF == 1 and boton[0] == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0                       # Modifica el estado global FFD
# ===== Fin control botón derecho =====

    if boton[1] == 1:
        pyautogui.click(button=tecla, clicks=2) 

    return FF

    
FFD = FFI = 0
def mouse_virtual(x_frame, y_frame , boton_izq, boton_der,scroll,estado_scroll,sensibilidad,mano):
    global FFD,FFI

    pyautogui.scroll(100*scroll)
    
    width_pantalla,height_pantalla = pyautogui.size()

   
    y = (int)((y_frame - 1/2*height_frame)*sensibilidad*1.1*height_pantalla/height_frame)
   
    if mano == 0:
        x = (int)((x_frame - 1/2*width_frame)*sensibilidad*1.1*width_pantalla/width_frame)        
    else:
        x = (int)((x_frame)*sensibilidad*1.1*width_pantalla/width_frame)
        
        
    # ... (Control de movimiento del cursor)
    if (x > 0) and (y > 0):
        try:
            pyautogui.moveTo(x, y)
        except pyautogui.FailSafeException:
            print("Movimiento detenido: mouse llegó a la esquina")
    else:
        pyautogui.moveTo(5, 5)

    if estado_scroll == 0:
        FFD = f_tecla(FFD,boton_der,'right')
        FFI = f_tecla(FFI,boton_izq,'left')


# ==============================
# Bucle principal con MediaPipe Hands
# ==============================
with mp_hands.Hands(
    static_image_mode=False,        # Procesa video en vivo
    max_num_hands=1,                # Máximo 1 mano detectada
    min_detection_confidence=0.7) as hands:  # Confianza mínima para detección

    while True:
        ret, frame = cap.read()     # Leer un frame de la cámara
        if ret == False:            # Si no hay frame, salir
            break

        height_frame, width_frame, _ = frame.shape   # Obtener dimensiones del frame

        coord_mano = Deteccion_mano_openCV(frame)   # Procesar frame con MediaPipe para detectar manos

        BD, BI, mouse_x, mouse_y = Deteccion_gestos(coord_mano)
        scroll,estado_scroll = mouse_gesto_operacion_scroll(coord_mano)

        mano = 0
        sensibilidad = 2
        mouse_virtual(mouse_x ,mouse_y , BD, BI, scroll,estado_scroll,sensibilidad,mano) 
        
        if keyboard.is_pressed("esc"):
            print("Saliste del bucle.")
            break
        #time.sleep(0.05)
# ==============================
# Liberar recursos
# ==============================
cap.release()           # Liberar la cámara
cv2.destroyAllWindows() # Cerrar todas las ventanas de OpenCV
# barra que modifique la frecuencia de corte

Saliste del bucle.


In [ ]:
# ==============================
# Importación de librerías
# ==============================
import cv2              # Librería OpenCV para visión por computadora
import mediapipe as mp  # Librería MediaPipe para detección de manos
import numpy as np      # Librería NumPy para cálculos numéricos
import pyautogui        # Librería para controlar el mouse y teclado
import time
import keyboard
import scipy.signal as sig
# ==============================
# Configuración de MediaPipe y cámara
# ==============================


def f_tecla(FF,boton,tecla):
# ===== Control botón derecho (usa FFD) =====
    if FF == 0 and boton == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1                       # Modifica el estado global FFD

    if FF == 1 and boton == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0                       # Modifica el estado global FFD
    # ===== Fin control botón derecho =====
    return FF

FFD = FFI = 0
def mouse_virtual(x_frame, y_frame , boton_izq, boton_der, scroll):
    global FFD,FFI
    width_pantalla,height_pantalla = pyautogui.size()

    pyautogui.scroll(scroll)

    FFD = f_tecla(FFD,boton_der,'right')
    FFI = f_tecla(FFI,boton_izq,'left')
    


# ==============================
# Función de prueba
# ==============================
def campo_de_prueba_mouse_virtual(FFD,FFI):
    x = 25                          # Coordenada X fija para prueba
    y = 25                          # Coordenada Y fija para prueba
    scroll = 10                  # Define si sube o baja la ruedita y cuanto 
    
    ubicacion = (x, y)              # Empaqueta coordenadas en una tupla
    senal = [0, 1, 0, 0, 0, 0, 0, 0, 0]  # Señal de prueba: presionar y soltar botón izquierdo
    
    time.sleep(5)                   # Espera 5 segundos antes de iniciar (para dar tiempo al usuario)
    print("doble click")            # Muestra texto (aunque en realidad hace clic único con espera larga)
    
    for i in range(len(senal)):     # Recorre cada valor en la lista señal
        boton = senal[i]            # Extrae el valor actual (0 = soltar, 1 = presionar)
        mouse_virtual(50, 50, boton, 0, scroll)  # Llama a mouse_virtual (botón derecho siempre 0)
        time.sleep(5)               # Espera 5 segundos entre cada acción


# ==============================
# Ejecución del programa
# ==============================
campo_de_prueba_mouse_virtual(FFD,FFI)   # Llama a la función de prueba y arranca el script



In [ ]:
!pip install PyQt5

In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QPushButton

app = QApplication.instance()
if not app:
    app = QApplication(sys.argv)

ventana = QWidget()
ventana.setWindowTitle("Mi ventana")

boton = QPushButton("Click", ventana)
boton.move(150,120)

ventana.resize(400,300)
ventana.show()

# Ejecutar aplicación
app.exec_()

In [ ]:
# ==============================
# Importación de librerías
# ==============================
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import time
import keyboard
import scipy.signal as sig

from PyQt5.QtWidgets import *
from PyQt5.QtCore import *

# ==============================
# Variables controladas por GUI
# ==============================
ejecutando = True

sensibilidad = 2
mano = 0
fc = 4

# ==============================
# Configuración de MediaPipe
# ==============================
mp_hands = mp.solutions.hands
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

window_size = 1000
positions = []

Fs = 30
orden = 1

b, a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Ventana PyQt
# ==============================

class Ventana(QWidget):

    def __init__(self):

        super().__init__()

        self.setWindowTitle("Control Mouse por Gestos")

        layout = QVBoxLayout()

        # MANO
        layout.addWidget(QLabel("Mano"))

        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")

        self.radio_der.setChecked(True)

        self.radio_der.toggled.connect(self.cambiar_mano)

        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))

        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setMinimum(1)
        self.slider_sens.setMaximum(5)
        self.slider_sens.setValue(2)

        self.slider_sens.valueChanged.connect(self.cambiar_sens)

        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))

        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setMinimum(1)
        self.slider_fc.setMaximum(10)
        self.slider_fc.setValue(4)

        self.slider_fc.valueChanged.connect(self.cambiar_fc)

        layout.addWidget(self.slider_fc)

        self.setLayout(layout)


        # BOTON DETENER
        self.boton_detener = QPushButton("DETENER PROGRAMA")
        
        self.boton_detener.clicked.connect(self.detener)
        
        layout.addWidget(self.boton_detener)


    def detener(self):
        global ejecutando    
        ejecutando = False

    def cambiar_mano(self):
        global mano
        if self.radio_der.isChecked():
            mano = 0
        else:
            mano = 1

    def cambiar_sens(self,val):
        global sensibilidad
        sensibilidad = val

    def cambiar_fc(self,val):
        global fc,b,a
        fc = val
        b,a = sig.butter(orden, fc/(0.5*Fs), btype='low')

    def closeEvent(self, event):
        global ejecutando    
        ejecutando = False
        event.accept()

# ==============================
# Filtro suavizado
# ==============================
def suavizar_pos(x, y):

    positions.append((x,y))

    if len(positions) > window_size:
        positions.pop(0)

    xs,ys = zip(*positions)

    x0_filtered = sig.lfilter(b,a,xs)
    y0_filtered = sig.lfilter(b,a,ys)

    return x0_filtered[-1],y0_filtered[-1]
    
# ==============================
# Scroll
# ==============================
estado_scroll = 0
valor_anterior_y = 0

def mouse_gesto_operacion_scroll(coord_mano):

    global estado_scroll,valor_anterior_y

    scroll = 0

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and estado_scroll == 0
            gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and estado_scroll == 0
            gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

            if gesto_estado_subir:
                estado_scroll = 1

            if gesto_estado_bajar:
                estado_scroll = 2

            if gesto_estado_reposo:
                estado_scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 1

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = -1

    else:
        estado_scroll = 0

    return scroll,estado_scroll

# ==============================
# Posición mouse
# ==============================
def mouse_ubicacion(coord_mano):

    mouse_x = mouse_y = 100

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)

            mouse_x,mouse_y = suavizar_pos(mouse_x,mouse_y)

    return mouse_x,mouse_y

# ==============================
# Gestos click
# ==============================
estadoBD2 = estadoBI2 = 0
def mouse_gesto_operacion(coord_mano):
    BI = (0,0)
    BD = (0,0)

    global estadoBD2,estadoBI2

    if coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            BD1 = (hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y)
            BI1 = (hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y)

            BD2 = (hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y)
            BI2 = (hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y)

            BD = (BD1,BD2)
            BI = (BI1,BI2)
    return BD,BI


# ==============================
# Detección mano
# ==============================
def Deteccion_mano_openCV(frame):
    frame = cv2.flip(frame,1)
    frame_rgb = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    coord_mano = hands.process(frame_rgb)

    return coord_mano


# ==============================
# Detección gestos
# ==============================
def Deteccion_gestos(coord_mano):
    mouse_x,mouse_y = mouse_ubicacion(coord_mano)
    BD,BI = mouse_gesto_operacion(coord_mano)

    return BD,BI,mouse_x,mouse_y


# ==============================
# Clicks mouse
# ==============================
FFD = FFI = 0
def f_tecla(FF,boton,tecla):
    if FF == 0 and boton[0] == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1

    if FF == 1 and boton[0] == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0

    if boton[1] == 1:
        pyautogui.click(button=tecla,clicks=2)

    return FF



# ==============================
# Mouse virtual
# ==============================
def mouse_virtual(x_frame,y_frame,boton_izq,boton_der,scroll,estado_scroll,sensibilidad,mano):
    global FFD,FFI

    pyautogui.scroll(100*scroll)
    width_pantalla,height_pantalla = pyautogui.size()

    y = int((y_frame - height_frame/2)*sensibilidad*height_pantalla/height_frame)

    if mano == 0:
        x = int((x_frame - width_frame/2)*sensibilidad*width_pantalla/width_frame)
    else:
        x = int((x_frame)*sensibilidad*width_pantalla/width_frame)

    if (x>0) and (y>0):
        try:
            pyautogui.moveTo(x,y)
        except pyautogui.FailSafeException:
            pass

    if estado_scroll == 0:
        FFD = f_tecla(FFD,boton_der,'right')
        FFI = f_tecla(FFI,boton_izq,'left')


# ==============================
# Inicializar GUI
# ==============================
app = QApplication(sys.argv)

ventana = Ventana()
ventana.show()

# ==============================
# LOOP PRINCIPAL
# ==============================
# ... (todo tu código anterior igual hasta el loop principal)

# ==============================
# LOOP PRINCIPAL
# ==============================
with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7) as hands:

    while ejecutando:
        app.processEvents() # Procesa eventos de la GUI (clics, movimiento)
        
        ret, frame = cap.read()
        if not ret:
            break

        height_frame, width_frame, _ = frame.shape
        coord_mano = Deteccion_mano_openCV(frame)
        BD, BI, mouse_x, mouse_y = Deteccion_gestos(coord_mano)
        scroll, estado_scroll = mouse_gesto_operacion_scroll(coord_mano)
        mouse_virtual(mouse_x, mouse_y, BD, BI, scroll, estado_scroll, sensibilidad, mano)

    # AL SALIR DEL WHILE
    cap.release()
    cv2.destroyAllWindows()
    # Esto asegura que MediaPipe libere los recursos antes de que la app muera
    hands.close() 

# Importante para limpiar el proceso de Qt
# sys.exit(app.exec_()) # Si usas esto, ponlo al final
        

# ==============================
# Cerrar
# ==============================
cap.release()
cv2.destroyAllWindows()


In [3]:
#PARA IMAGENES DE SCROLL

import cv2
import mediapipe as mp
import numpy as np
import pyautogui

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

color_mouse_pointer = (255, 0, 255)

# Puntos de la pantalla-juego
SCREEN_GAME_X_INI = 150
SCREEN_GAME_Y_INI = 160
SCREEN_GAME_X_FIN = 150 + 780
SCREEN_GAME_Y_FIN = 160 + 450

aspect_ratio_screen = (SCREEN_GAME_X_FIN - SCREEN_GAME_X_INI) / (SCREEN_GAME_Y_FIN - SCREEN_GAME_Y_INI)
print("aspect_ratio_screen:", aspect_ratio_screen)

X_Y_INI = 100

def calculate_distance(x1, y1, x2, y2):
    p1 = np.array([x1, y1])
    p2 = np.array([x2, y2])
    return np.linalg.norm(p1 - p2)


def detect_finger_down(hand_landmarks):
    finger_down = False
    color_base = (255, 0, 112)
    color_index = (255, 0, 255)

    #posición “lateral” o “neutra” y “vertical”
    x_index = int(hand_landmarks.landmark[8].x * width)
    y_index = int(hand_landmarks.landmark[8].y * height)   
    #cv2.circle(output, (x_index, y_index), 5, color_index, 2)

    x1 = int(hand_landmarks.landmark[13].x * width)
    y1 = int(hand_landmarks.landmark[13].y * height)       
    #cv2.circle(output, (x1, y1), 5, color_index, 2)

    #posición “pronacion” TERMINAR
    x_index = int(hand_landmarks.landmark[8].x * width)
    y_index = int(hand_landmarks.landmark[8].y * height)   
    #cv2.circle(output, (x_index, y_index), 5, color_index, 2)

    x1 = int(hand_landmarks.landmark[5].x * width)
    y1 = int(hand_landmarks.landmark[5].y * height)       
   # cv2.circle(output, (x1, y1), 5, color_index, 2)



    #posición “REPOSO” TERMINAR
    x_index = int(hand_landmarks.landmark[0].x * width)
    y_index = int(hand_landmarks.landmark[0].y * height)   
    cv2.circle(output, (x_index, y_index), 5, color_index, 2)

    x1 = int(hand_landmarks.landmark[9].x * width)
    y1 = int(hand_landmarks.landmark[9].y * height)       
    cv2.circle(output, (x1, y1), 5, color_index, 2)

    x2 = int(hand_landmarks.landmark[17].x * width)
    y2 = int(hand_landmarks.landmark[17].y * height)       
    cv2.circle(output, (x2, y2), 5, color_index, 2)

    x3 = int(hand_landmarks.landmark[5].x * width)
    y3 = int(hand_landmarks.landmark[5].y * height)       
    cv2.circle(output, (x3, y3), 5, color_index, 2)

            #gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and estado_scroll == 0
            #gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and estado_scroll == 0
            #gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)


    

    return finger_down



with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5) as hands:

    while True:
        ret, frame = cap.read()
        if ret == False:
            break

        height, width, _ = frame.shape
        frame = cv2.flip(frame, 1)

        # Dibujando un área proporcional a la del juego
        area_width = width - X_Y_INI * 2
        area_height = int(area_width / aspect_ratio_screen)
        aux_image = np.zeros(frame.shape, np.uint8)
        aux_image = cv2.rectangle(aux_image, (X_Y_INI, X_Y_INI), (X_Y_INI + area_width, X_Y_INI +area_height), (255, 0, 0), -1)
        output = cv2.addWeighted(frame, 1, aux_image, 0.7, 0)

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = hands.process(frame_rgb)
        
        if results.multi_hand_landmarks is not None:
            for hand_landmarks in results.multi_hand_landmarks:
                x = int(hand_landmarks.landmark[9].x * width)
                y = int(hand_landmarks.landmark[9].y * height)
                xm = np.interp(x, (X_Y_INI, X_Y_INI + area_width), (SCREEN_GAME_X_INI, SCREEN_GAME_X_FIN))
                ym = np.interp(y, (X_Y_INI, X_Y_INI + area_height), (SCREEN_GAME_Y_INI, SCREEN_GAME_Y_FIN))
                pyautogui.moveTo(int(xm), int(ym))
                if detect_finger_down(hand_landmarks):
                    pyautogui.click()
                

        #cv2.imshow('Frame', frame)
        cv2.imshow('output', output)
        if cv2.waitKey(1) & 0xFF == 27:
            break
cap.release()
cv2.destroyAllWindows()

aspect_ratio_screen: 1.7333333333333334


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pyautogui

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
fps = cap.get(cv2.CAP_PROP_FPS)
print(fps)

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *

# Desactivar el failsafe de pyautogui para evitar cierres inesperados si el mouse toca una esquina
pyautogui.FAILSAFE = False

# ==============================
# HILO DE PROCESAMIENTO (Lógica de Cámara y Gestos)
# ==============================
class MouseThread(QThread):
    def __init__(self):
        super().__init__()
        self.ejecutando = True
        
        # Variables de control (sincronizadas con la GUI)
        self.sensibilidad = 2
        self.mano = 0  # 0: Derecha, 1: Izquierda
        self.fc = 4
        self.Fs = 30
        self.orden = 1
        
        # Variables internas de estado
        self.positions = []
        self.window_size = 20
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        self.FFD = 0
        self.FFI = 0
        
        # Inicializar filtro
        self.actualizar_filtro()

    def actualizar_filtro(self):
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        xs, ys = zip(*self.positions)
        x_filt = sig.lfilter(self.b, self.a, xs)
        y_filt = sig.lfilter(self.b, self.a, ys)
        return x_filt[-1], y_filt[-1]

    def gestionar_clicks(self, FF, boton, tecla):
        # boton[0] es click simple, boton[1] es doble click
        if FF == 0 and boton[0] == 1:
            pyautogui.mouseDown(button=tecla)
            FF = 1
        if FF == 1 and boton[0] == 0:
            pyautogui.mouseUp(button=tecla)
            FF = 0
        if boton[1] == 1:
            pyautogui.click(button=tecla, clicks=2)
        return FF

    def run(self):
        cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
        mp_hands = mp.solutions.hands
        
        with mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7) as hands:

            while self.ejecutando:
                ret, frame = cap.read()
                if not ret: break

                h_f, w_f, _ = frame.shape
                frame = cv2.flip(frame, 1)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = hands.process(frame_rgb)

                if results.multi_hand_landmarks:
                    for hl in results.multi_hand_landmarks:
                        # 1. POSICION MOUSE (Landmark 0)
                        mx_raw = int(hl.landmark[0].x * w_f)
                        my_raw = int(hl.landmark[0].y * h_f)
                        mx, my = self.suavizar_pos(mx_raw, my_raw)

                        # 2. GESTOS CLICKS
                        # BD: (Anular, Meñique) -> Click Derecho
                        # BI: (Índice, Medio) -> Click Izquierdo
                        BD = (hl.landmark[8].y > hl.landmark[5].y, hl.landmark[16].y > hl.landmark[13].y)
                        BI = (hl.landmark[12].y > hl.landmark[9].y, hl.landmark[20].y > hl.landmark[17].y)

                        # 3. SCROLL
                        scroll = 0
                        # Lógica simplificada de scroll basada en tu código original
                        if (abs(hl.landmark[8].x - hl.landmark[13].x) * w_f < 30) and self.estado_scroll == 0:
                            self.estado_scroll = 1 # Modo Subir
                        elif (abs(hl.landmark[5].y - hl.landmark[8].y) * h_f < 50) and self.estado_scroll == 0:
                            self.estado_scroll = 2 # Modo Bajar
                        
                        # Reposo para resetear scroll
                        if (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x):
                            self.estado_scroll = 0

                        if self.estado_scroll != 0:
                            if abs(hl.landmark[8].y - self.valor_anterior_y) * h_f > 5:
                                scroll = 1 if hl.landmark[8].y > self.valor_anterior_y else -1
                                self.valor_anterior_y = hl.landmark[8].y

                        # 4. MOVER MOUSE VIRTUAL
                        sw, sh = pyautogui.size()
                        pyautogui.scroll(100 * scroll)
                        
                        y_p = int((my - h_f/2) * self.sensibilidad * sh / h_f)
                        if self.mano == 0: # Derecha
                            x_p = int((mx - w_f/2) * self.sensibilidad * sw / w_f)
                        else: # Izquierda
                            x_p = int((mx) * self.sensibilidad * sw / w_f)

                        if x_p > 0 and y_p > 0:
                            pyautogui.moveTo(x_p, y_p, _pause=False)

                        if self.estado_scroll == 0:
                            self.FFD = self.gestionar_clicks(self.FFD, BD, 'right')
                            self.FFI = self.gestionar_clicks(self.FFI, BI, 'left')
                else:
                    self.estado_scroll = 0

        cap.release()
        cv2.destroyAllWindows()

    def stop(self):
        self.ejecutando = False
        self.wait()

# ==============================
# VENTANA PRINCIPAL (GUI)
# ==============================
class Ventana(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(300, 350)

        layout = QVBoxLayout()

        # MANO
        layout.addWidget(QLabel("<b>Seleccionar Mano:</b>"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("<b>Sensibilidad:</b>"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(2)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("<b>Suavizado (Filtro):</b>"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 15)
        self.slider_fc.setValue(4)
        layout.addWidget(self.slider_fc)

        # BOTON DETENER
        self.boton_detener = QPushButton("DETENER PROGRAMA")
        self.boton_detener.setStyleSheet("background-color: #ff4444; color: white; font-weight: bold;")
        layout.addWidget(self.boton_detener)

        self.setLayout(layout)

        # Iniciar hilo de procesamiento
        self.hilo = MouseThread()
        self.hilo.start()

        # Conectar señales
        self.radio_der.toggled.connect(self.actualizar_config)
        self.slider_sens.valueChanged.connect(self.actualizar_config)
        self.slider_fc.valueChanged.connect(self.actualizar_config)
        self.boton_detener.clicked.connect(self.close)

    def actualizar_config(self):
        self.hilo.mano = 0 if self.radio_der.isChecked() else 1
        self.hilo.sensibilidad = self.slider_sens.value()
        self.hilo.fc = self.slider_fc.value()
        self.hilo.actualizar_filtro()

    def closeEvent(self, event):
        print("Cerrando cámara de forma segura...")
        self.hilo.stop()
        event.accept()

# ==============================
# EJECUCIÓN
# ==============================
if __name__ == '__main__':
    app = QApplication(sys.argv)
    ventana = Ventana()
    ventana.show()
    sys.exit(app.exec_())

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import time
import keyboard
import scipy.signal as sig

from PyQt5.QtWidgets import QApplication, QWidget, QPushButton, QVBoxLayout, QSlider#nuevo
from PyQt5.QtCore import QThread,Qt#nuevo

# ==============================
# Programa
# ==============================
class Worker(QThread):#nuevo

    def __init__(self):#nuevo
        
        super().__init__()
        self.pausado = False
        self.mano_boton = "Right"
        self.sensibilidad = 5
        
    def toggle_pause(self):  #nuevo
        self.pausado = not self.pausado
    
    def run(self):

        mp_hands = mp.solutions.hands
        cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

        window_size = 1000
        positions = []

        Fs = 100000
        fc = 4000
        orden = 1
        b, a = sig.butter(orden, fc/(0.5*Fs), btype='low', analog=False, output='ba')

        FFD = 0
        FFI = 0

        def suavizar_pos(x, y):
            positions.append((x, y))
            if len(positions) > window_size:
                positions.pop(0)

            xs, ys = zip(*positions)
            x0_filtered = sig.lfilter(b, a, xs)
            y0_filtered = sig.lfilter(b, a, ys)

            return x0_filtered[-1], y0_filtered[-1]
            
        def seleccionar_mano(coord_mano, mano_deseada):#nuevo

            if coord_mano.multi_hand_landmarks is None:
                return None
        
            for hand_landmarks, handedness in zip(
                    coord_mano.multi_hand_landmarks,
                    coord_mano.multi_handedness):
        
                etiqueta = handedness.classification[0].label
        
                if etiqueta == mano_deseada:
                    return hand_landmarks
        
            return None
        def mouse_ubicacion(hand_landmarks, width_frame, height_frame):

            mouse_x = mouse_y = 100
        
            if hand_landmarks is not None:
        
                mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
                mouse_y = int(hand_landmarks.landmark[0].y * height_frame)
        
                mouse_x, mouse_y = suavizar_pos(mouse_x, mouse_y)
        
            return mouse_x, mouse_y
            
        def mouse_gesto_operacion(hand_landmarks):

            BI = BD = 0
        
            if hand_landmarks is not None:
        
                BD = hand_landmarks.landmark[8].y > hand_landmarks.landmark[6].y
                BI = hand_landmarks.landmark[12].y > hand_landmarks.landmark[10].y
        
            return BD, BI

        def f_tecla(FF, boton, tecla):
            if FF == 0 and boton == 1:
                pyautogui.mouseDown(button=tecla)
                FF = 1
            if FF == 1 and boton == 0:
                pyautogui.mouseUp(button=tecla)
                FF = 0
            return FF

        def mouse_virtual(x_frame, y_frame, boton_izq, boton_der, width_frame, height_frame):
            nonlocal FFD, FFI

            width_pantalla, height_pantalla = pyautogui.size()

            x = int((x_frame - 0.5*width_frame)*2.2*width_pantalla/width_frame)
            y = int((y_frame - 0.5*height_frame)*2.2*height_pantalla/height_frame)

            if (x > 0) and (y > 0):
                try:
                    pyautogui.moveTo(x, y)
                except pyautogui.FailSafeException:
                    pass
            else:
                pyautogui.moveTo(5, 5)

            FFD = f_tecla(FFD, boton_der, 'right')
            FFI = f_tecla(FFI, boton_izq, 'left')


        with mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7) as hands:

            while self.isRunning():

                if self.pausado:
                    time.sleep(0.1)
                    continue
                    
                ret, frame = cap.read()
                if not ret:
                    break

                height_frame, width_frame, _ = frame.shape

                frame = cv2.flip(frame, 1)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                coord_mano = hands.process(frame_rgb)

                mano = seleccionar_mano(coord_mano, self.mano_boton)
                
                mouse_x, mouse_y = mouse_ubicacion(mano, width_frame, height_frame)
                BD, BI = mouse_gesto_operacion(mano)
                
                mouse_virtual(mouse_x, mouse_y, BD, BI, width_frame, height_frame)

                if keyboard.is_pressed("esc"):
                    break

                time.sleep(0.05)

        cap.release()
        cv2.destroyAllWindows()


# ==============================
# VENTANA DEL MENU
# ==============================
class Ventana(QWidget):

    def __init__(self):
        super().__init__()

        self.setWindowTitle("Mouse Virtual")
        self.resize(500, 350)

        self.boton_inicio = QPushButton("Correr")
        self.boton_inicio.setFixedSize(150, 50)

        self.boton_mano = QPushButton("Derecha")
        self.boton_mano.setFixedSize(150, 40)

        self.slider_sensibilidad = QSlider(Qt.Horizontal)
        self.slider_sensibilidad.setMinimum(1)
        self.slider_sensibilidad.setMaximum(10)
        self.slider_sensibilidad.setValue(5)
        
        layout = QVBoxLayout()

        layout.addStretch()
        layout.addWidget(self.boton_inicio, alignment=Qt.AlignCenter)
        layout.addWidget(self.boton_mano, alignment=Qt.AlignCenter)
        layout.addWidget(self.slider_sensibilidad)
        layout.addStretch()
        
        self.setLayout(layout)
        
        self.worker = Worker()
        
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        self.boton_mano.clicked.connect(self.cambiar_mano)
        self.slider_sensibilidad.valueChanged.connect(self.cambiar_sensibilidad)

        self.mano_derecha = True

      

    def iniciar_programa(self):

        if not self.worker.isRunning():
            self.worker.start()
            self.boton_inicio.setText("Pausar")

        else:
            self.worker.toggle_pause()

            if self.worker.pausado:
                self.boton_inicio.setText("Continuar")
            else:
                self.boton_inicio.setText("Pausar")


    def cambiar_mano(self):

        self.mano_derecha = not self.mano_derecha

        if self.mano_derecha:
            self.boton_mano.setText("Derecha")
            self.worker.mano_boton = "Right"
        else:
            self.boton_mano.setText("Izquierda")
            self.worker.mano_boton = "Left"

    def closeEvent(self, event):

        if self.worker.isRunning():
            self.worker.terminate()
            self.worker.wait()

        event.accept()
        
    def cambiar_sensibilidad(self, valor):

        self.worker.sensibilidad = valor
        
# ==============================
# EJECUCIÓN DE LA APP
# ==============================
if __name__ == "__main__":
    app = QApplication(sys.argv)
    ventana = Ventana()
    ventana.show()
    sys.exit(app.exec_())

In [ ]:
# ==============================
# Importación de librerías
# ==============================
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import time
import keyboard
import scipy.signal as sig

from PyQt5.QtWidgets import *
from PyQt5.QtCore import *

# ==============================
# Variables controladas por GUI
# ==============================

sensibilidad = 2
mano = 0
fc = 4
ejecutando = False

# ==============================
# Configuración de MediaPipe
# ==============================

mp_hands = mp.solutions.hands
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

window_size = 1000
positions = []

Fs = 30
orden = 1

b, a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Ventana PyQt
# ==============================

class Ventana(QWidget):

    def __init__(self):

        super().__init__()

        self.setWindowTitle("Control Mouse por Gestos")

        layout = QVBoxLayout()

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))

        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")

        self.radio_der.setChecked(True)

        self.radio_der.toggled.connect(self.cambiar_mano)

        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))

        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setMinimum(1)
        self.slider_sens.setMaximum(10)
        self.slider_sens.setValue(2)

        self.slider_sens.valueChanged.connect(self.cambiar_sens)

        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))

        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setMinimum(1)
        self.slider_fc.setMaximum(10)
        self.slider_fc.setValue(4)

        self.slider_fc.valueChanged.connect(self.cambiar_fc)

        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    def iniciar_programa(self):

        global ejecutando

        ejecutando = not ejecutando

        if ejecutando:
            self.boton_inicio.setText("Pausar")
        else:
            self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):

        global mano

        if self.radio_der.isChecked():
            mano = 0
        else:
            mano = 1

    def cambiar_sens(self,val):

        global sensibilidad

        sensibilidad = val

    def cambiar_fc(self,val):

        global fc,b,a

        fc = val
        b,a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Filtro suavizado
# ==============================

def suavizar_pos(x, y):

    positions.append((x,y))

    if len(positions) > window_size:
        positions.pop(0)

    xs,ys = zip(*positions)

    x0_filtered = sig.lfilter(b,a,xs)
    y0_filtered = sig.lfilter(b,a,ys)

    return x0_filtered[-1],y0_filtered[-1]

# ==============================
# Scroll
# ==============================

estado_scroll = 0
valor_anterior_y = 0

def mouse_gesto_operacion_scroll(coord_mano):

    global estado_scroll,valor_anterior_y

    scroll = 0

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and estado_scroll == 0
            gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and estado_scroll == 0
            gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

            if gesto_estado_subir:
                estado_scroll = 1

            if gesto_estado_bajar:
                estado_scroll = 2

            if gesto_estado_reposo:
                estado_scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 1

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = -1

    else:

        estado_scroll = 0

    return scroll,estado_scroll

# ==============================
# Posición mouse
# ==============================
def mouse_ubicacion(coord_mano):

    mouse_x = mouse_y = 100

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)

            mouse_x,mouse_y = suavizar_pos(mouse_x,mouse_y)

    return mouse_x,mouse_y

# ==============================
# Gestos click
# ==============================
estadoBD2 = estadoBI2 = 0

def mouse_gesto_operacion(coord_mano):

    BI = (0,0)
    BD = (0,0)

    global estadoBD2,estadoBI2

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            BD1 = (hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y)
            BI1 = (hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y)

            BD2 = (hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y)
            BI2 = (hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y)

            BD = (BD1,BD2)
            BI = (BI1,BI2)

    return BD,BI

# ==============================
# Detección mano
# ==============================

def Deteccion_mano_openCV(frame):

    frame = cv2.flip(frame,1)

    frame_rgb = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)

    coord_mano = hands.process(frame_rgb)

    return coord_mano

# ==============================
# Detección gestos
# ==============================

def Deteccion_gestos(coord_mano):

    mouse_x,mouse_y = mouse_ubicacion(coord_mano)

    BD,BI = mouse_gesto_operacion(coord_mano)

    return BD,BI,mouse_x,mouse_y

# ==============================
# Clicks mouse
# ==============================

def f_tecla(FF,boton,tecla):

    if FF == 0 and boton[0] == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1

    if FF == 1 and boton[0] == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0

    if boton[1] == 1:
        pyautogui.click(button=tecla,clicks=2)

    return FF

FFD = FFI = 0

# ==============================
# Mouse virtual
# ==============================

def mouse_virtual(x_frame,y_frame,boton_izq,boton_der,scroll,estado_scroll,sensibilidad,mano):

    global FFD,FFI

    pyautogui.scroll(100*scroll)

    width_pantalla,height_pantalla = pyautogui.size()

    y = int((y_frame - height_frame/2)*sensibilidad*height_pantalla/height_frame)

    if mano == 0:
        x = int((x_frame - width_frame/2)*sensibilidad*width_pantalla/width_frame)
    else:
        x = int((x_frame)*sensibilidad*width_pantalla/width_frame)

    if (x>0) and (y>0):

        try:
            pyautogui.moveTo(x,y)
        except pyautogui.FailSafeException:
            pass

    if estado_scroll == 0:

        FFD = f_tecla(FFD,boton_der,'right')
        FFI = f_tecla(FFI,boton_izq,'left')

# ==============================
# Inicializar GUI
# ==============================

app = QApplication(sys.argv)

ventana = Ventana()
ventana.show()

# ==============================
# LOOP PRINCIPAL
# ==============================

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7) as hands:

    while True:

        app.processEvents()

        if not ejecutando:
            continue

        ret,frame = cap.read()

        if ret == False:
            break

        height_frame,width_frame,_ = frame.shape

        coord_mano = Deteccion_mano_openCV(frame)

        BD,BI,mouse_x,mouse_y = Deteccion_gestos(coord_mano)

        scroll,estado_scroll = mouse_gesto_operacion_scroll(coord_mano)

        mouse_virtual(mouse_x ,mouse_y , BD, BI, scroll,estado_scroll,sensibilidad,mano)

        if keyboard.is_pressed("esc"):            
            break

# ==============================
# Cerrar
# ==============================
cap.release()
cv2.destroyAllWindows()


In [ ]:
# ==============================
# Importación de librerías
# ==============================
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import time
import keyboard
import scipy.signal as sig

from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap


# ==============================
# Variables controladas por GUI
# ==============================

sensibilidad = 2
mano = 0
fc = 4
ejecutando = False

# ==============================
# Configuración de MediaPipe
# ==============================

mp_hands = mp.solutions.hands
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

window_size = 1000
positions = []

Fs = 30
orden = 2

b, a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Ventana PyQt
# ==============================

class Ventana(QWidget):

    def __init__(self):

        super().__init__()

        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(500,450)

        layout = QVBoxLayout()


        imagen = QLabel()        

        pixmap = QPixmap("mouse virtual logo.jpg")
        
        imagen.setPixmap(pixmap.scaled(200,150))

        imagen.setAlignment(Qt.AlignCenter)
        
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))

        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")

        self.radio_der.setChecked(True)

        self.radio_der.toggled.connect(self.cambiar_mano)

        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))

        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setMinimum(1)
        self.slider_sens.setMaximum(10)
        self.slider_sens.setValue(2)

        self.slider_sens.valueChanged.connect(self.cambiar_sens)

        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))

        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setMinimum(1)
        self.slider_fc.setMaximum(10)
        self.slider_fc.setValue(4)

        self.slider_fc.valueChanged.connect(self.cambiar_fc)

        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    def iniciar_programa(self):

        global ejecutando

        ejecutando = not ejecutando

        if ejecutando:
            self.boton_inicio.setText("Pausar")
        else:
            self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):

        global mano

        if self.radio_der.isChecked():
            mano = 0
        else:
            mano = 1

    def cambiar_sens(self,val):

        global sensibilidad
        sensibilidad = val

    def cambiar_fc(self,val):

        global fc,b,a

        fc = val
        b,a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Filtro suavizado
# ==============================

def suavizar_pos(x, y):

    positions.append((x,y))

    if len(positions) > window_size:
        positions.pop(0)

    xs,ys = zip(*positions)

    x0_filtered = sig.lfilter(b,a,xs)
    y0_filtered = sig.lfilter(b,a,ys)

    return x0_filtered[-1],y0_filtered[-1]

# ==============================
# Scroll
# ==============================

estado_scroll = 0
valor_anterior_y = 0

def mouse_gesto_operacion_scroll(coord_mano):

    global estado_scroll,valor_anterior_y

    scroll = 0

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and estado_scroll == 0
            gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and estado_scroll == 0
            gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

            if gesto_estado_subir:
                estado_scroll = 1

            if gesto_estado_bajar:
                estado_scroll = 2

            if gesto_estado_reposo:
                estado_scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 1

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = -1

    else:

        estado_scroll = 0

    return scroll,estado_scroll

# ==============================
# Posición mouse
# ==============================

def mouse_ubicacion(coord_mano):

    mouse_x = mouse_y = 100

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)

            mouse_x,mouse_y = suavizar_pos(mouse_x,mouse_y)

    return mouse_x,mouse_y

# ==============================
# Gestos click
# ==============================

estadoBD2 = estadoBI2 = 0

def mouse_gesto_operacion(coord_mano):

    BI = (0,0)
    BD = (0,0)

    global estadoBD2,estadoBI2

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            BD1 = (hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y)
            BI1 = (hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y)

            BD2 = (hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y)
            BI2 = (hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y)

            BD = (BD1,BD2)
            BI = (BI1,BI2)

    return BD,BI

# ==============================
# Detección mano
# ==============================

def Deteccion_mano_openCV(frame):

    frame = cv2.flip(frame,1)
    frame_rgb = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)

    coord_mano = hands.process(frame_rgb)

    return coord_mano

# ==============================
# Detección gestos
# ==============================

def Deteccion_gestos(coord_mano):

    mouse_x,mouse_y = mouse_ubicacion(coord_mano)

    BD,BI = mouse_gesto_operacion(coord_mano)

    return BD,BI,mouse_x,mouse_y

# ==============================
# Clicks mouse
# ==============================

def f_tecla(FF,boton,tecla):

    if FF == 0 and boton[0] == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1

    if FF == 1 and boton[0] == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0

    if boton[1] == 1:
        pyautogui.click(button=tecla,clicks=2)

    return FF

FFD = FFI = 0

# ==============================
# Mouse virtual
# ==============================

def mouse_virtual(x_frame,y_frame,boton_izq,boton_der,scroll,estado_scroll,sensibilidad,mano):

    global FFD,FFI

    pyautogui.scroll(100*scroll)

    width_pantalla,height_pantalla = pyautogui.size()

    y = int((y_frame - height_frame/2)*sensibilidad*height_pantalla/height_frame)

    if mano == 0:
        x = int((x_frame - width_frame/2)*sensibilidad*width_pantalla/width_frame)
    else:
        x = int((x_frame)*sensibilidad*width_pantalla/width_frame)

    if (x>0) and (y>0):

        try:
            pyautogui.moveTo(x,y)
        except pyautogui.FailSafeException:
            pass

    if estado_scroll == 0:

        FFD = f_tecla(FFD,boton_der,'right')
        FFI = f_tecla(FFI,boton_izq,'left')

# ==============================
# Inicializar GUI
# ==============================

app = QApplication(sys.argv)

ventana = Ventana()
ventana.show()

# ==============================
# LOOP PRINCIPAL
# ==============================

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7) as hands:

    while True:

        app.processEvents()

        if not ejecutando:
            time.sleep(0.05)   # 🔹 mejora CPU
            continue

        ret,frame = cap.read()

        if ret == False:
            break

        frame = cv2.resize(frame, (640, 480))#

        height_frame,width_frame,_ = frame.shape

        coord_mano = Deteccion_mano_openCV(frame)

        BD,BI,mouse_x,mouse_y = Deteccion_gestos(coord_mano)

        scroll,estado_scroll = mouse_gesto_operacion_scroll(coord_mano)

        mouse_virtual(mouse_x ,mouse_y , BD, BI, scroll,estado_scroll,sensibilidad,mano)

        if keyboard.is_pressed("esc") or not ventana.isVisible():
            break

# ==============================
# Cerrar
# ==============================

cap.release()
cv2.destroyAllWindows()

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Desactivar el failsafe de pyautogui si prefieres que no crashee en las esquinas
pyautogui.FAILSAFE = False

class VirtualMouse(QWidget):
    def __init__(self):
        super().__init__()
        
        # --- Variables de Control ---
        self.ejecutando = False
        self.sensibilidad = 2
        self.mano_seleccionada = 0 # 0: Derecha, 1: Izquierda
        self.fc = 4
        self.Fs = 30
        self.orden = 1
        self.positions = []
        self.window_size = 5 # Reducido para menor latencia
        
        # Filtro inicial
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')
        
        # MediaPipe Setup
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7,
            model_complexity=0 # 0 es más rápido, 1 es más preciso
        )
        
        self.cap = None
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_frame)
        
        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Control Mouse por Gestos Pro")
        self.setFixedSize(400, 500)
        layout = QVBoxLayout()

        # Logo / Status
        self.label_status = QLabel("Estado: Detenido")
        self.label_status.setAlignment(Qt.AlignCenter)
        layout.addWidget(self.label_status)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar Cámara")
        self.boton_inicio.setMinimumHeight(50)
        self.boton_inicio.clicked.connect(self.toggle_programa)
        layout.addWidget(self.boton_inicio)

        # Configuración de Mano
        layout.addWidget(QLabel("Seleccionar Mano:"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # Deslizadores
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 15)
        self.slider_sens.setValue(self.sensibilidad)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        layout.addWidget(QLabel("Suavizado (Frecuencia de Corte)"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(self.fc)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    # --- Lógica de Control ---
    def toggle_programa(self):
        if not self.ejecutando:
            self.cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se pudo acceder a la cámara")
                return
            self.ejecutando = True
            self.timer.start(30) # ~30 FPS
            self.boton_inicio.setText("Detener Cámara")
            self.label_status.setText("Estado: Ejecutando")
        else:
            self.ejecutando = False
            self.timer.stop()
            if self.cap:
                self.cap.release()
            self.boton_inicio.setText("Iniciar Cámara")
            self.label_status.setText("Estado: Detenido")

    def cambiar_mano(self):
        self.mano_seleccionada = 0 if self.radio_der.isChecked() else 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        
        xs, ys = zip(*self.positions)
        # El filtro lfilter necesita un historial, para pocos datos 
        # a veces es mejor un promedio simple o lfilter con padding
        xf = sig.lfilter(self.b, self.a, xs)[-1]
        yf = sig.lfilter(self.b, self.a, ys)[-1]
        return xf, yf

    def procesar_frame(self):
        ret, frame = self.cap.read()
        if not ret: return

        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resultados = self.hands.process(rgb_frame)

        if resultados.multi_hand_landmarks:
            hand_landmarks = resultados.multi_hand_landmarks[0]
            
            # 1. Obtener Posición (Landmark 9 es el centro de la palma, más estable que el 0)
            raw_x = hand_landmarks.landmark[9].x * w
            raw_y = hand_landmarks.landmark[9].y * h
            
            fx, fy = self.suavizar_pos(raw_x, raw_y)
            
            # 2. Mover Mouse
            sw, sh = pyautogui.size()
            # Mapeo con sensibilidad
            nx = int((fx - w/4) * (sw / (w/2)) * (self.sensibilidad/2))
            ny = int((fy - h/4) * (sh / (h/2)) * (self.sensibilidad/2))
            
            try:
                pyautogui.moveTo(nx, ny, _pause=False)
            except: pass

            # 3. Lógica de Clicks (Simplificada)
            # Índice abajo -> Click Izquierdo
            if hand_landmarks.landmark[8].y > hand_landmarks.landmark[6].y:
                pyautogui.click()

    def closeEvent(self, event):
        if self.cap:
            self.cap.release()
        event.accept()

if __name__ == '__main__':
    app = QApplication(sys.argv)
    mouse_app = VirtualMouse()
    mouse_app.show()
    sys.exit(app.exec_())

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Configuración de PyAutoGUI
pyautogui.FAILSAFE = False

class Ventana(QWidget):
    def __init__(self):
        super().__init__()

        # --- Variables de Control ---
        self.ejecutando = False
        self.sensibilidad = 2
        self.mano_seleccionada = 0  # 0: Derecha, 1: Izquierda
        self.fc = 4
        self.Fs = 30
        self.orden = 2
        self.positions = []
        self.window_size = 15
        
        # Variables de estado para clicks y scroll
        self.FFD = 0
        self.FFI = 0
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        
        # --- MediaPipe Setup ---
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        
        # --- Filtro Inicial ---
        self.actualizar_filtros()

        # --- Timer para el loop (reemplaza al while True) ---
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_loop)
        
        self.cap = cv2.VideoCapture() # Se inicializa vacío

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(500, 450)
        layout = QVBoxLayout()

        # Imagen/Logo
        imagen = QLabel()
        pixmap = QPixmap("mouse virtual logo.jpg")
        if not pixmap.isNull():
            imagen.setPixmap(pixmap.scaled(200, 150, Qt.KeepAspectRatio))
        imagen.setAlignment(Qt.AlignCenter)
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.setMinimumHeight(40)
        self.boton_inicio.clicked.connect(self.toggle_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(2)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro (Suavizado)"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(4)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    # --- Lógica de Interfaz ---
    def toggle_programa(self):
        if not self.ejecutando:
            # Intentar abrir la cámara solo al iniciar
            self.cap.open(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se puede acceder a la cámara")
                return
            
            self.ejecutando = True
            self.boton_inicio.setText("Pausar / Apagar Cámara")
            self.timer.start(30) # ~30 FPS
            self.positions = [] # Limpiar historial de filtro
        else:
            self.apagar_camara()

    def apagar_camara(self):
        self.ejecutando = False
        self.timer.stop()
        if self.cap.isOpened():
            self.cap.release()
        self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):
        self.mano_seleccionada = 0 if self.radio_der.isChecked() else 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.actualizar_filtros()

    def actualizar_filtros(self):
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    # --- Procesamiento de Imagen y Gestos ---
    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        
        xs, ys = zip(*self.positions)
        if len(xs) > self.orden * 2:
            xf = sig.lfilter(self.b, self.a, xs)[-1]
            yf = sig.lfilter(self.b, self.a, ys)[-1]
            return xf, yf
        return x, y

    def procesar_loop(self):
        # Verificar tecla de escape
        if keyboard.is_pressed("esc"):
            self.close()
            return

        ret, frame = self.cap.read()
        if not ret:
            self.apagar_camara()
            return

        frame = cv2.resize(frame, (640, 480))
        h_frame, w_frame, _ = frame.shape
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        res = self.hands.process(frame_rgb)

        if res.multi_hand_landmarks:
            hand_landmarks = res.multi_hand_landmarks[0] # Una sola mano
            
            # 1. Posición del Mouse (Landmark 0)
            mx = int(hand_landmarks.landmark[0].x * w_frame)
            my = int(hand_landmarks.landmark[0].y * h_frame)
            mx, my = self.suavizar_pos(mx, my)

            # 2. Gestos de Click
            # BD = (Anular, Meñique), BI = (Indice, Medio)
            # True si el dedo está "cerrado" (punta por debajo de nudillo)
            bi1 = hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y
            bd1 = hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y
            bi2 = hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y
            bd2 = hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y
            
            # 3. Gesto de Scroll
            scroll, est_scroll = self.calcular_scroll(hand_landmarks, w_frame, h_frame)

            # 4. Ejecutar Movimiento Virtual
            self.ejecutar_mouse_virtual(mx, my, (bd1, bd2), (bi1, bi2), 
                                        scroll, est_scroll, w_frame, h_frame)

    def calcular_scroll(self, hand_landmarks, w, h):
        scroll = 0
        # Gesto subir: Indice y Medio pegados horizontalmente
        gesto_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*w < 30) and self.estado_scroll == 0
        # Gesto bajar: Indice muy cerca de su base
        gesto_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*h < 50) and self.estado_scroll == 0
        # Reposo: basado en distancia palma
        gesto_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

        if gesto_subir: self.estado_scroll = 1
        if gesto_bajar: self.estado_scroll = 2
        if gesto_reposo: self.estado_scroll = 0

        # Detección de movimiento vertical del índice para scroll
        curr_y = hand_landmarks.landmark[8].y
        if abs(curr_y - self.valor_anterior_y)*h > 5:
            if self.estado_scroll == 1:
                scroll = 1 if curr_y > self.valor_anterior_y else 0
            elif self.estado_scroll == 2:
                scroll = 0 if curr_y > self.valor_anterior_y else -1
            self.valor_anterior_y = curr_y

        return scroll, self.estado_scroll

    def ejecutar_mouse_virtual(self, x_f, y_f, boton_der, boton_izq, scroll, est_scroll, w_f, h_f):
        # Scroll
        if scroll != 0:
            pyautogui.scroll(100 * scroll)

        # Movimiento
        sw, sh = pyautogui.size()
        
        # Mapeo de coordenadas
        y = int((y_f - h_f/2) * self.sensibilidad * sh / h_f)
        if self.mano_seleccionada == 0: # Derecha
            x = int((x_f - w_f/2) * self.sensibilidad * sw / w_f)
        else: # Izquierda
            x = int(x_f * self.sensibilidad * sw / w_f)

        if x > 0 and y > 0:
            try:
                pyautogui.moveTo(x, y, _pause=False)
            except:
                pass

        # Clicks (solo si no se está haciendo scroll)
        if est_scroll == 0:
            self.FFD = self.f_tecla(self.FFD, boton_der, 'right')
            self.FFI = self.f_tecla(self.FFI, boton_izq, 'left')

    def f_tecla(self, FF, boton, tecla):
        # boton[0] es click simple, boton[1] es doble click
        if FF == 0 and boton[0]:
            pyautogui.mouseDown(button=tecla)
            FF = 1
        elif FF == 1 and not boton[0]:
            pyautogui.mouseUp(button=tecla)
            FF = 0
        
        if boton[1]:
            pyautogui.click(button=tecla, clicks=2)
        return FF

    def closeEvent(self, event):
        self.apagar_camara()
        event.accept()

# --- Ejecución ---
if __name__ == "__main__":
    app = QApplication(sys.argv)
    ventana = Ventana()
    ventana.show()
    sys.exit(app.exec_())

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Configuración de PyAutoGUI
pyautogui.FAILSAFE = False

class Ventana(QWidget):
    def __init__(self):
        super().__init__()

        # ==============================
        # Variables controladas por GUI (Tus variables originales)
        # ==============================
        self.sensibilidad = 2
        self.mano = 0
        self.fc = 4
        self.ejecutando = False
        
        # Filtro y posiciones
        self.window_size = 1000
        self.positions = []
        self.Fs = 30
        self.orden = 2
        self.b = None
        self.a = None
        self.actualizar_filtros() # Inicializa b y a

        # Variables de estado para clicks y scroll
        self.FFD = 0
        self.FFI = 0
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        
        # ==============================
        # Configuración de MediaPipe
        # ==============================
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        
        # Timer para reemplazar el while True y no bloquear la GUI
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_loop)
        
        self.cap = cv2.VideoCapture() 

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(500, 450)
        layout = QVBoxLayout()

        # Logo
        imagen = QLabel()
        pixmap = QPixmap("mouse virtual logo.jpg")
        if not pixmap.isNull():
            imagen.setPixmap(pixmap.scaled(200, 150, Qt.KeepAspectRatio))
        imagen.setAlignment(Qt.AlignCenter)
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.setMinimumHeight(40)
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(self.sensibilidad)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(self.fc)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    # ==============================
    # Lógica de Interfaz (Tus funciones de Ventana)
    # ==============================
    def iniciar_programa(self):
        self.ejecutando = not self.ejecutando

        if self.ejecutando:
            self.cap.open(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se puede acceder a la cámara")
                self.ejecutando = False
                return
            
            self.boton_inicio.setText("Pausar")
            self.timer.start(30) 
            self.positions = [] 
        else:
            self.apagar_recursos()

    def apagar_recursos(self):
        self.ejecutando = False
        self.timer.stop()
        if self.cap.isOpened():
            self.cap.release()
        self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):
        if self.radio_der.isChecked():
            self.mano = 0
        else:
            self.mano = 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.actualizar_filtros()

    def actualizar_filtros(self):
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    # ==============================
    # Filtro suavizado
    # ==============================
    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        
        xs, ys = zip(*self.positions)
        if len(xs) > self.orden * 2:
            x0_filtered = sig.lfilter(self.b, self.a, xs)
            y0_filtered = sig.lfilter(self.b, self.a, ys)
            return x0_filtered[-1], y0_filtered[-1]
        return x, y

    # ==============================
    # Loop Principal Corregido
    # ==============================
    def procesar_loop(self):
        if keyboard.is_pressed("esc"):
            self.close()
            return

        ret, frame = self.cap.read()
        if not ret:
            self.apagar_recursos()
            return

        frame = cv2.resize(frame, (640, 480))
        height_frame, width_frame, _ = frame.shape
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        coord_mano = self.hands.process(frame_rgb)

        if coord_mano.multi_hand_landmarks:
            hand_landmarks = coord_mano.multi_hand_landmarks[0]
            
            # 1. Posición mouse
            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)
            mouse_x, mouse_y = self.suavizar_pos(mouse_x, mouse_y)

            # 2. Gestos click (BD1, BI1, BD2, BI2)
            bi1 = hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y
            bd1 = hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y
            bi2 = hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y
            bd2 = hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y
            
            # 3. Scroll
            scroll, est_scroll = self.mouse_gesto_operacion_scroll(hand_landmarks, width_frame, height_frame)

            # 4. Mouse virtual
            self.mouse_virtual(mouse_x, mouse_y, (bd1, bd2), (bi1, bi2), 
                               scroll, est_scroll, width_frame, height_frame)

    # ==============================
    # Scroll
    # ==============================
    def mouse_gesto_operacion_scroll(self, hand_landmarks, width_frame, height_frame):
        scroll = 0
        gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and self.estado_scroll == 0
        gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and self.estado_scroll == 0
        gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

        if gesto_estado_subir: self.estado_scroll = 1
        if gesto_estado_bajar: self.estado_scroll = 2
        if gesto_estado_reposo: self.estado_scroll = 0

        curr_y = hand_landmarks.landmark[8].y
        if abs(curr_y - self.valor_anterior_y)*height_frame > 5:
            if self.estado_scroll == 1:
                scroll = 1 if curr_y > self.valor_anterior_y else 0
            elif self.estado_scroll == 2:
                scroll = 0 if curr_y > self.valor_anterior_y else -1
            self.valor_anterior_y = curr_y

        return scroll, self.estado_scroll

    # ==============================
    # Mouse virtual (Lógica de movimiento y Clicks)
    # ==============================
    def mouse_virtual(self, x_frame, y_frame, boton_der, boton_izq, scroll, estado_scroll, width_frame, height_frame):
        if scroll != 0:
            pyautogui.scroll(100 * scroll)

        width_pantalla, height_pantalla = pyautogui.size()
        
        y = int((y_frame - height_frame/2) * self.sensibilidad * height_pantalla / height_frame)
        if self.mano == 0: # Derecha
            x = int((x_frame - width_frame/2) * self.sensibilidad * width_pantalla / width_frame)
        else: # Izquierda
            x = int(x_frame * self.sensibilidad * width_pantalla / width_frame)

        if x > 0 and y > 0:
            try:
                pyautogui.moveTo(x, y, _pause=False)
            except:
                pass

        if estado_scroll == 0:
            self.FFD = self.f_tecla(self.FFD, boton_der, 'right')
            self.FFI = self.f_tecla(self.FFI, boton_izq, 'left')

    def f_tecla(self, FF, boton, tecla):
        if FF == 0 and boton[0]:
            pyautogui.mouseDown(button=tecla)
            FF = 1
        elif FF == 1 and not boton[0]:
            pyautogui.mouseUp(button=tecla)
            FF = 0
        
        if boton[1]:
            pyautogui.click(button=tecla, clicks=2)
        return FF

    def closeEvent(self, event):
        self.apagar_recursos()
        event.accept()

# --- Ejecución ---
if __name__ == "__main__":
    app = QApplication(sys.argv)
    ventana = Ventana()
    ventana.show()
    app.exec_()

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Configuración de PyAutoGUI
pyautogui.FAILSAFE = False

class Ventana(QWidget):
    def __init__(self):
        super().__init__()

        # ==============================
        # Variables controladas por GUI
        # ==============================
        self.sensibilidad = 2
        self.mano = 0
        self.fc = 4
        self.ejecutando = False
        
        # Filtro y posiciones
        self.window_size = 1000
        self.positions = []
        self.Fs = 30
        self.orden = 2
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

        # Variables de estado para clicks y scroll
        self.FFD = 0
        self.FFI = 0
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        
        # ==============================
        # Configuración de MediaPipe
        # ==============================
        self.mp_hands = mp.solutions.hands
        # Se inicializa aquí para que persista durante la vida de la ventana
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        
        self.cap = cv2.VideoCapture() 
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_loop)

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Mouse Virtual")
        self.setFixedSize(500, 450)
        layout = QVBoxLayout()

        # Logo
        imagen = QLabel()
        pixmap = QPixmap("mouse virtual logo.jpg")
        if not pixmap.isNull():
            imagen.setPixmap(pixmap.scaled(200, 150, Qt.KeepAspectRatio))
        imagen.setAlignment(Qt.AlignCenter)
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.setMinimumHeight(40)
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(self.sensibilidad)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(self.fc)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    def iniciar_programa(self):
        self.ejecutando = not self.ejecutando

        if self.ejecutando:
            # CAP_DSHOW es más estable en Windows para evitar que la cámara tarde en encender
            self.cap.open(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se puede acceder a la cámara")
                self.ejecutando = False
                return
            
            self.boton_inicio.setText("Pausar")
            self.timer.start(30) 
            self.positions = [] 
        else:
            self.apagar_recursos()

    def apagar_recursos(self):
        """Libera la cámara y detiene el timer sin cerrar la app"""
        self.ejecutando = False
        self.timer.stop()
        if self.cap.isOpened():
            self.cap.release()
        self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):
        self.mano = 0 if self.radio_der.isChecked() else 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        
        xs, ys = zip(*self.positions)
        if len(xs) > self.orden * 2:
            x0_filtered = sig.lfilter(self.b, self.a, xs)
            y0_filtered = sig.lfilter(self.b, self.a, ys)
            return x0_filtered[-1], y0_filtered[-1]
        return x, y

    def procesar_loop(self):
        # Esc para cerrar rápido
        if keyboard.is_pressed("esc"):
            self.close()
            return

        ret, frame = self.cap.read()
        if not ret:
            return

        frame = cv2.resize(frame, (640, 480))
        height_frame, width_frame, _ = frame.shape
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        coord_mano = self.hands.process(frame_rgb)

        if coord_mano.multi_hand_landmarks:
            hand_landmarks = coord_mano.multi_hand_landmarks[0]
            
            # 1. Posición mouse (Usando Landmark 0)
            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)
            mouse_x, mouse_y = self.suavizar_pos(mouse_x, mouse_y)

            # 2. Gestos click
            bi1 = hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y
            bd1 = hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y
            bi2 = hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y
            bd2 = hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y
            
            # 3. Scroll
            scroll, est_scroll = self.mouse_gesto_operacion_scroll(hand_landmarks, width_frame, height_frame)

            # 4. Mouse virtual
            self.mouse_virtual(mouse_x, mouse_y, (bd1, bd2), (bi1, bi2), 
                               scroll, est_scroll, width_frame, height_frame)

    def mouse_gesto_operacion_scroll(self, hand_landmarks, width_frame, height_frame):
        scroll = 0
        gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and self.estado_scroll == 0
        gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and self.estado_scroll == 0
        gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

        if gesto_estado_subir: self.estado_scroll = 1
        if gesto_estado_bajar: self.estado_scroll = 2
        if gesto_estado_reposo: self.estado_scroll = 0

        curr_y = hand_landmarks.landmark[8].y
        if abs(curr_y - self.valor_anterior_y)*height_frame > 5:
            if self.estado_scroll == 1:
                scroll = 1 if curr_y > self.valor_anterior_y else 0
            elif self.estado_scroll == 2:
                scroll = 0 if curr_y > self.valor_anterior_y else -1
            self.valor_anterior_y = curr_y

        return scroll, self.estado_scroll

    def mouse_virtual(self, x_frame, y_frame, boton_der, boton_izq, scroll, estado_scroll, width_frame, height_frame):
        if scroll != 0:
            pyautogui.scroll(100 * scroll)

        width_pantalla, height_pantalla = pyautogui.size()
        
        y = int((y_frame - height_frame/2) * self.sensibilidad * height_pantalla / height_frame)
        if self.mano == 0:
            x = int((x_frame - width_frame/2) * self.sensibilidad * width_pantalla / width_frame)
        else:
            x = int(x_frame * self.sensibilidad * width_pantalla / width_frame)

        if x > 0 and y > 0:
            try:
                pyautogui.moveTo(x, y, _pause=False)
            except:
                pass

        if estado_scroll == 0:
            self.FFD = self.f_tecla(self.FFD, boton_der, 'right')
            self.FFI = self.f_tecla(self.FFI, boton_izq, 'left')

    def f_tecla(self, FF, boton, tecla):
        if FF == 0 and boton[0]:
            pyautogui.mouseDown(button=tecla)
            FF = 1
        elif FF == 1 and not boton[0]:
            pyautogui.mouseUp(button=tecla)
            FF = 0
        
        if boton[1]:
            pyautogui.click(button=tecla, clicks=2)
        return FF

    def closeEvent(self, event):
        """Este método se ejecuta al cerrar la ventana (X)"""
        self.apagar_recursos()
        # IMPORTANTE: Liberar MediaPipe explícitamente para Jupyter
        if hasattr(self, 'hands'):
            self.hands.close()
        cv2.destroyAllWindows()
        event.accept()

# ==============================
# Ejecución segura para Jupyter
# ==============================
if __name__ == "__main__":
    # Si ya existe una instancia de app, la usamos (evita crasheos en Jupyter)
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    
    try:
        ventana = Ventana()
        ventana.show()
        app.exec_()
    finally:
        # Esto se ejecuta siempre, incluso si hay un error o detienes el Kernel
        cv2.destroyAllWindows()

In [ ]:
!pip show PyQt5

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# ==========================================
# CONFIGURACIÓN Y VARIABLES GLOBALES
# ==========================================
pyautogui.FAILSAFE = False

# Variables de Control
sensibilidad = 2
mano = 0
fc = 4
ejecutando = False
window_size_filtro = 1000
positions = []
Fs = 30.0
orden = 2
b, a = None, None

# Variables de FPS
tiempo_previo = time.time()
tiempos_fps = []
ventana_promedio = 100 

# Variables de clicks y scroll
FFD = 0
FFI = 0
estado_scroll = 0
valor_anterior_y = 0

# MediaPipe y Cámara
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==========================================
# FUNCIONES DE LÓGICA
# ==========================================

def actualizar_coeficientes_filtro():
    global b, a
    fs_segura = max(Fs, fc * 2.1) 
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')

def suavizar_pos(x, y):
    global positions
    positions.append((x, y))
    if len(positions) > window_size_filtro:
        positions.pop(0)
    
    if len(positions) > orden * 2:
        xs, ys = zip(*positions)
        x0_filt = sig.lfilter(b, a, xs)
        y0_filt = sig.lfilter(b, a, ys)
        return x0_filt[-1], y0_filt[-1]
    return x, y

def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]: 
        pyautogui.mouseDown(button=tecla)
        return 1
    elif FF_val == 1 and not boton[0]: 
        pyautogui.mouseUp(button=tecla)
        return 0
    if boton[1]: 
        pyautogui.click(button=tecla, clicks=2)
    return FF_val

def mouse_gesto_operacion_scroll(hand_landmarks, w_f, h_f):
    global estado_scroll, valor_anterior_y
    scroll = 0
    gesto_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*w_f < 30) and estado_scroll == 0
    gesto_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*h_f < 50) and estado_scroll == 0
    gesto_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

    if gesto_subir: estado_scroll = 1
    if gesto_bajar: estado_scroll = 2
    if gesto_reposo: estado_scroll = 0

    curr_y = hand_landmarks.landmark[8].y
    if abs(curr_y - valor_anterior_y)*h_f > 5:
        if estado_scroll == 1: scroll = 1 if curr_y > valor_anterior_y else 0
        elif estado_scroll == 2: scroll = 0 if curr_y > valor_anterior_y else -1
        valor_anterior_y = curr_y
    return scroll, estado_scroll

def procesar_loop():
    global tiempo_previo, Fs, FFD, FFI
    
    t_actual = time.time()
    dt = t_actual - tiempo_previo
    tiempo_previo = t_actual

    if dt > 0: tiempos_fps.append(dt)
    if len(tiempos_fps) > ventana_promedio: tiempos_fps.pop(0)

    if len(tiempos_fps) == ventana_promedio:
        Fs = 1 / (sum(tiempos_fps) / len(tiempos_fps))
        actualizar_coeficientes_filtro()
        print(f"FPS: {Fs:.2f}", end="\r")

    if keyboard.is_pressed("esc"):
        ventana.close()
        return

    ret, frame = cap.read()
    if not ret: return

    frame = cv2.resize(frame, (640, 480))
    h_frame, w_frame, _ = frame.shape
    frame = cv2.flip(frame, 1)
    res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if res.multi_hand_landmarks:
        hl = res.multi_hand_landmarks[0]
        mx, my = suavizar_pos(int(hl.landmark[0].x * w_frame), int(hl.landmark[0].y * h_frame))

        # Dedos
        bi1, bd1 = hl.landmark[8].y > hl.landmark[5].y, hl.landmark[12].y > hl.landmark[9].y
        bi2, bd2 = hl.landmark[16].y > hl.landmark[13].y, hl.landmark[20].y > hl.landmark[17].y
        
        scroll, est_scroll = mouse_gesto_operacion_scroll(hl, w_frame, h_frame)
        
        # Mouse Virtual
        if scroll != 0: pyautogui.scroll(100 * scroll)
        sw, sh = pyautogui.size()
        y_px = int((my - h_frame/2) * sensibilidad * sh / h_frame)
        x_px = int((mx - w_frame/2) * sensibilidad * sw / w_frame) if mano == 0 else int(mx * sensibilidad * sw / w_frame)

        if x_px > 0 and y_px > 0:
            try: pyautogui.moveTo(x_px, y_px, _pause=False)
            except: pass

        if est_scroll == 0:
            FFD = f_tecla(FFD, (bd1, bd2), 'right')
            FFI = f_tecla(FFI, (bi1, bi2), 'left')

# ==========================================
# FUNCIONES DE INTERFAZ (UI)
# ==========================================

def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    ejecutando = not ejecutando
    if ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            QMessageBox.critical(None, "Error", "No hay cámara")
            ejecutando = False
            return
        boton_inicio.setText("Pausar")
        timer.start(30)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()

def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): cap.release()
    boton_inicio.setText("Iniciar")

def al_cerrar(event):
    apagar_recursos()
    hands.close()
    cv2.destroyAllWindows()
    event.accept()

# ==========================================
# SETUP DE LA APLICACIÓN
# ==========================================
actualizar_coeficientes_filtro()

app = QApplication(sys.argv)
ventana = QWidget()
ventana.setWindowTitle("Control Mouse por Gestos")
ventana.setFixedSize(500, 450)
ventana.closeEvent = al_cerrar
layout = QVBoxLayout()

# UI Elements
label_logo = QLabel()
px = QPixmap("mouse virtual logo.jpg")
if not px.isNull(): label_logo.setPixmap(px.scaled(200, 150, Qt.KeepAspectRatio))
label_logo.setAlignment(Qt.AlignCenter)

boton_inicio = QPushButton("Iniciar")
boton_inicio.setMinimumHeight(40)
boton_inicio.clicked.connect(iniciar_programa)

radio_der = QRadioButton("Derecha")
radio_izq = QRadioButton("Izquierda")
radio_der.setChecked(True)

def cambio_mano(): global mano; mano = 0 if radio_der.isChecked() else 1
radio_der.toggled.connect(cambio_mano)

slider_sens = QSlider(Qt.Horizontal)
slider_sens.setRange(1, 10)
slider_sens.setValue(sensibilidad)
slider_sens.valueChanged.connect(lambda v: globals().update(sensibilidad=v))

slider_fc = QSlider(Qt.Horizontal)
slider_fc.setRange(1, 10)
slider_fc.setValue(fc)
def cambio_fc(v): global fc; fc = v; actualizar_coeficientes_filtro()
slider_fc.valueChanged.connect(cambio_fc)

# Agregar al layout
layout.addWidget(label_logo)
layout.addWidget(boton_inicio)
layout.addWidget(QLabel("Mano"))
layout.addWidget(radio_der)
layout.addWidget(radio_izq)
layout.addWidget(QLabel("Sensibilidad"))
layout.addWidget(slider_sens)
layout.addWidget(QLabel("Frecuencia corte filtro"))
layout.addWidget(slider_fc)

ventana.setLayout(layout)
timer.timeout.connect(procesar_loop)

ventana.show()
app.exec_()

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Configuración de PyAutoGUI
pyautogui.FAILSAFE = False

class Ventana(QWidget):
    def __init__(self):
        super().__init__()

        # ==============================
        # Variables de Control
        # ==============================
        self.sensibilidad = 2
        self.mano = 0
        self.fc = 4
        self.ejecutando = False
        
        # Filtro y posiciones
        self.window_size_filtro = 1000
        self.positions = []
        self.Fs = 30  # Valor inicial por defecto
        self.orden = 2
        
        # Inicializamos los coeficientes del filtro
        self.actualizar_coeficientes_filtro()

        # Variables para FPS Estables
        self.tiempo_previo = time.time()
        self.tiempos_fps = []
        self.ventana_promedio = 20 
        
        # Variables de clicks y scroll
        self.FFD = 0
        self.FFI = 0
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        
        # ==============================
        # Configuración de MediaPipe
        # ==============================
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        
        self.cap = cv2.VideoCapture() 
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_loop)

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(500, 450)
        layout = QVBoxLayout()

        # Logo
        imagen = QLabel()
        pixmap = QPixmap("mouse virtual logo.jpg")
        if not pixmap.isNull():
            imagen.setPixmap(pixmap.scaled(200, 150, Qt.KeepAspectRatio))
        imagen.setAlignment(Qt.AlignCenter)
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.setMinimumHeight(40)
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(self.sensibilidad)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(self.fc)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    def actualizar_coeficientes_filtro(self):
        # Evitamos que Fs sea demasiado baja para el cálculo
        fs_segura = max(self.Fs, self.fc * 2.1) 
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5 * fs_segura), btype='low')

    def iniciar_programa(self):
        self.ejecutando = not self.ejecutando
        if self.ejecutando:
            self.cap.open(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se puede acceder a la cámara")
                self.ejecutando = False
                return
            self.boton_inicio.setText("Pausar")
            self.timer.start(30) 
            self.positions = [] 
            self.tiempos_fps = [] 
            self.tiempo_previo = time.time()
        else:
            self.apagar_recursos()

    def apagar_recursos(self):
        self.ejecutando = False
        self.timer.stop()
        if self.cap.isOpened():
            self.cap.release()
        self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):
        self.mano = 0 if self.radio_der.isChecked() else 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.actualizar_coeficientes_filtro()

    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size_filtro:
            self.positions.pop(0)
        xs, ys = zip(*self.positions)
        if len(xs) > self.orden * 2:
            x0_filtered = sig.lfilter(self.b, self.a, xs)
            y0_filtered = sig.lfilter(self.b, self.a, ys)
            return x0_filtered[-1], y0_filtered[-1]
        return x, y

    def procesar_loop(self):
        # --- Cálculo de FPS Promediados ---
        tiempo_actual = time.time()
        delta_t = tiempo_actual - self.tiempo_previo
        self.tiempo_previo = tiempo_actual

        if delta_t > 0:
            self.tiempos_fps.append(delta_t)
        if len(self.tiempos_fps) > self.ventana_promedio:
            self.tiempos_fps.pop(0)

        if len(self.tiempos_fps) > 0:
            fps_promedio = 1 / (sum(self.tiempos_fps) / len(self.tiempos_fps))
            
            # Actualizamos self.Fs dinámicamente cada 20 cuadros para que el filtro se adapte
            if len(self.tiempos_fps) == self.ventana_promedio:
                self.Fs = fps_promedio
                self.actualizar_coeficientes_filtro()
            
            print(f"FPS Promedio: {fps_promedio:.2f} | Fs Filtro: {self.Fs:.2f} ", end="\r")

        if keyboard.is_pressed("esc"):
            self.close()
            return

        ret, frame = self.cap.read()
        if not ret: return

        frame = cv2.resize(frame, (640, 480))
        h_frame, w_frame, _ = frame.shape
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        coord_mano = self.hands.process(frame_rgb)

        if coord_mano.multi_hand_landmarks:
            hand_landmarks = coord_mano.multi_hand_landmarks[0]
            mx = int(hand_landmarks.landmark[0].x * w_frame)
            my = int(hand_landmarks.landmark[0].y * h_frame)
            mx, my = self.suavizar_pos(mx, my)

            bi1 = hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y
            bd1 = hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y
            bi2 = hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y
            bd2 = hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y
            
            scroll, est_scroll = self.mouse_gesto_operacion_scroll(hand_landmarks, w_frame, h_frame)
            self.mouse_virtual(mx, my, (bd1, bd2), (bi1, bi2), scroll, est_scroll, w_frame, h_frame)

    def mouse_gesto_operacion_scroll(self, hand_landmarks, width_frame, height_frame):
        scroll = 0
        gesto_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and self.estado_scroll == 0
        gesto_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and self.estado_scroll == 0
        gesto_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

        if gesto_subir: self.estado_scroll = 1
        if gesto_bajar: self.estado_scroll = 2
        if gesto_reposo: self.estado_scroll = 0

        curr_y = hand_landmarks.landmark[8].y
        if abs(curr_y - self.valor_anterior_y)*height_frame > 5:
            if self.estado_scroll == 1: scroll = 1 if curr_y > self.valor_anterior_y else 0
            elif self.estado_scroll == 2: scroll = 0 if curr_y > self.valor_anterior_y else -1
            self.valor_anterior_y = curr_y
        return scroll, self.estado_scroll

    def mouse_virtual(self, x_f, y_f, b_der, b_izq, scroll, est_scroll, w_f, h_f):
        if scroll != 0: pyautogui.scroll(100 * scroll)
        sw, sh = pyautogui.size()
        y = int((y_f - h_f/2) * self.sensibilidad * sh / h_f)
        x = int((x_f - w_f/2) * self.sensibilidad * sw / w_f) if self.mano == 0 else int(x_f * self.sensibilidad * sw / w_f)

        if x > 0 and y > 0:
            try: pyautogui.moveTo(x, y, _pause=False)
            except: pass

        if est_scroll == 0:
            self.FFD = self.f_tecla(self.FFD, b_der, 'right')
            self.FFI = self.f_tecla(self.FFI, b_izq, 'left')

    def f_tecla(self, FF, boton, tecla):
        if FF == 0 and boton[0]: 
            pyautogui.mouseDown(button=tecla) 
            FF = 1
            
        elif FF == 1 and not boton[0]: 
            pyautogui.mouseUp(button=tecla)
            FF = 0
            
        if boton[1]: 
            pyautogui.click(button=tecla, clicks=2)
            
        return FF

    def closeEvent(self, event):
        self.apagar_recursos()
        if hasattr(self, 'hands'): self.hands.close()
        cv2.destroyAllWindows()
        event.accept()

if __name__ == "__main__":
    app = QApplication.instance() or QApplication(sys.argv)
    try:
        ventana = Ventana()
        ventana.show()
        app.exec_()
    finally:
        cv2.destroyAllWindows()

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# ==========================================
# CONFIGURACIÓN Y VARIABLES GLOBALES
# ==========================================
pyautogui.FAILSAFE = False

# Variables de Control
sensibilidad = 2
mano = 0
fc = 4
ejecutando = False
window_size_filtro = 1000
positions = []
Fs = 30.0
orden = 2
b, a = None, None

# Variables de FPS
tiempo_previo = time.time()
tiempos_fps = []
ventana_promedio = 50 

# Variables de clicks y scroll
FFD = 0
FFI = 0
estado_scroll = 0
valor_anterior_y = 0

# MediaPipe y Cámara
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==========================================
# FUNCIONES DE LÓGICA
# ==========================================

def actualizar_coeficientes_filtro():
    global b, a
    fs_segura = max(Fs, fc * 2.1) 
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')

def suavizar_pos(x, y):
    global positions
    positions.append((x, y))
    if len(positions) > window_size_filtro:
        positions.pop(0)
    
    if len(positions) > orden * 2:
        xs, ys = zip(*positions)
        x0_filt = sig.lfilter(b, a, xs)
        y0_filt = sig.lfilter(b, a, ys)
        return x0_filt[-1], y0_filt[-1]
    return x, y

def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]: 
        pyautogui.mouseDown(button=tecla)
        return 1
    elif FF_val == 1 and not boton[0]: 
        pyautogui.mouseUp(button=tecla)
        return 0
    if boton[1]: 
        pyautogui.click(button=tecla, clicks=2)
    return FF_val

def mouse_gesto_operacion_scroll(hand_landmarks, w_f, h_f):
    global estado_scroll, valor_anterior_y
    scroll = 0
    gesto_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*w_f < 30) and estado_scroll == 0
    gesto_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*h_f < 50) and estado_scroll == 0
    gesto_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

    if gesto_subir: estado_scroll = 1
    if gesto_bajar: estado_scroll = 2
    if gesto_reposo: estado_scroll = 0

    curr_y = hand_landmarks.landmark[8].y
    if abs(curr_y - valor_anterior_y)*h_f > 5:
        if estado_scroll == 1: scroll = 1 if curr_y > valor_anterior_y else 0
        elif estado_scroll == 2: scroll = 0 if curr_y > valor_anterior_y else -1
        valor_anterior_y = curr_y
    return scroll, estado_scroll

def procesar_loop():
    global tiempo_previo, Fs, FFD, FFI
    
    t_actual = time.time()
    dt = t_actual - tiempo_previo
    tiempo_previo = t_actual

    if dt > 0: tiempos_fps.append(dt)
    if len(tiempos_fps) > ventana_promedio: tiempos_fps.pop(0)

    if len(tiempos_fps) == ventana_promedio:
        Fs = 1 / (sum(tiempos_fps) / len(tiempos_fps))
        actualizar_coeficientes_filtro()
        print(f"FPS: {Fs:.2f}", end="\r")

    if keyboard.is_pressed("esc"):
        ventana.close()
        return

    ret, frame = cap.read()
    if not ret: return

    frame = cv2.resize(frame, (640, 480))
    h_frame, w_frame, _ = frame.shape
    frame = cv2.flip(frame, 1)
    res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if res.multi_hand_landmarks:
        hl = res.multi_hand_landmarks[0]
        mx, my = suavizar_pos(int(hl.landmark[0].x * w_frame), int(hl.landmark[0].y * h_frame))

        # Dedos
        bi1, bd1 = hl.landmark[8].y > hl.landmark[5].y, hl.landmark[12].y > hl.landmark[9].y
        bi2, bd2 = hl.landmark[16].y > hl.landmark[13].y, hl.landmark[20].y > hl.landmark[17].y
        
        scroll, est_scroll = mouse_gesto_operacion_scroll(hl, w_frame, h_frame)
        
        # Mouse Virtual
        if scroll != 0: pyautogui.scroll(100 * scroll)
        sw, sh = pyautogui.size()
        y_px = int((my - h_frame/2) * sensibilidad * sh / h_frame)
        x_px = int((mx - w_frame/2) * sensibilidad * sw / w_frame) if mano == 0 else int(mx * sensibilidad * sw / w_frame)

        if x_px > 0 and y_px > 0:
            try: pyautogui.moveTo(x_px, y_px, _pause=False)
            except: pass

        if est_scroll == 0:
            FFD = f_tecla(FFD, (bd1, bd2), 'right')
            FFI = f_tecla(FFI, (bi1, bi2), 'left')

# ==========================================
# FUNCIONES DE INTERFAZ (UI)
# ==========================================

def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    ejecutando = not ejecutando
    if ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            QMessageBox.critical(None, "Error", "No hay cámara")
            ejecutando = False
            return
        boton_inicio.setText("Pausar")
        timer.start(30)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()

def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): cap.release()
    boton_inicio.setText("Iniciar")

def al_cerrar(event):
    apagar_recursos()
    hands.close()
    cv2.destroyAllWindows()
    event.accept()

# ==========================================
# SETUP DE LA APLICACIÓN
# ==========================================
actualizar_coeficientes_filtro()

app = QApplication(sys.argv)
ventana = QWidget()
ventana.setWindowTitle("Control Mouse por Gestos")
ventana.setFixedSize(500, 450)
ventana.closeEvent = al_cerrar
layout = QVBoxLayout()

# UI Elements
label_logo = QLabel()
px = QPixmap("mouse virtual logo.jpg")
if not px.isNull(): label_logo.setPixmap(px.scaled(200, 150, Qt.KeepAspectRatio))
label_logo.setAlignment(Qt.AlignCenter)

boton_inicio = QPushButton("Iniciar")
boton_inicio.setMinimumHeight(40)
boton_inicio.clicked.connect(iniciar_programa)

radio_der = QRadioButton("Derecha")
radio_izq = QRadioButton("Izquierda")
radio_der.setChecked(True)

def cambio_mano(): global mano; mano = 0 if radio_der.isChecked() else 1
radio_der.toggled.connect(cambio_mano)

slider_sens = QSlider(Qt.Horizontal)
slider_sens.setRange(1, 10)
slider_sens.setValue(sensibilidad)
slider_sens.valueChanged.connect(lambda v: globals().update(sensibilidad=v))

slider_fc = QSlider(Qt.Horizontal)
slider_fc.setRange(1, 10)
slider_fc.setValue(fc)
def cambio_fc(v): global fc; fc = v; actualizar_coeficientes_filtro()
slider_fc.valueChanged.connect(cambio_fc)

# Agregar al layout
layout.addWidget(label_logo)
layout.addWidget(boton_inicio)
layout.addWidget(QLabel("Mano"))
layout.addWidget(radio_der)
layout.addWidget(radio_izq)
layout.addWidget(QLabel("Sensibilidad"))
layout.addWidget(slider_sens)
layout.addWidget(QLabel("Frecuencia corte filtro"))
layout.addWidget(slider_fc)

ventana.setLayout(layout)
timer.timeout.connect(procesar_loop)

ventana.show()
(app.exec_())

In [12]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# ==============================
# CONFIGURACIÓN GLOBAL
# ==============================
pyautogui.FAILSAFE = False

# Variables de Control
sensibilidad = 2
mano = 0
fc = 4
ejecutando = False
positions = []
window_size_filtro = 1000
Fs = 30.0
orden = 2
b, a = None, None

# Estado de Clicks y Scroll
FFD = 0
FFI = 0
estado_scroll = 0
valor_anterior_y = 0
tiempo_previo = time.time()
tiempos_fps = []

# Inicialización MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=1, 
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==============================
# LÓGICA DE PROCESAMIENTO
# ==============================
def actualizar_coeficientes():
    global b, a
    fs_segura = max(Fs, fc * 2.1)
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')


def suavizar_pos(x, y):
    global positions
    positions.append((x, y))
    if len(positions) > window_size_filtro: 
        positions.pop(0)
        
    if len(positions) > orden * 2:
        xs, ys = zip(*positions)
        return sig.lfilter(b, a, xs)[-1], sig.lfilter(b, a, ys)[-1]
    return x, y


def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]: 
        pyautogui.mouseDown(button=tecla)
        return 1
        
    elif FF_val == 1 and not boton[0]: 
        pyautogui.mouseUp(button=tecla)
        return 0
        
    if boton[1]: 
        pyautogui.click(button=tecla, clicks=2)
        
    return FF_val


def mouse_ubicacion(hl):
    mouse_x, mouse_y = int(hl.landmark[0].x * 640), int(hl.landmark[0].y * 480)
    mouse_x, mouse_y = suavizar_pos((mouse_x), (mouse_y))
    
    return mouse_x, mouse_y

def mouse_gesto_operacion(hl):
    bi1 = hl.landmark[8].y > hl.landmark[5].y    
    bi2 = hl.landmark[16].y > hl.landmark[13].y
    
    bd1 = hl.landmark[12].y > hl.landmark[9].y
    bd2 = hl.landmark[20].y > hl.landmark[17].y

    return (bi1, bi2),(bd1, bd2)


def Deteccion_mano_openCV(frame):
    frame = cv2.flip(cv2.resize(frame, (640, 480)), 1)
    coord_mano = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    return coord_mano


def Deteccion_gestos(hl):
    mouse_x, mouse_y = mouse_ubicacion(hl)
    BI,BD = mouse_gesto_operacion(hl)
    scroll,estado_scroll = mouse_gesto_operacion_scroll(hl)
    
    return BI,BD,mouse_x, mouse_y,scroll,estado_scroll


FFD=FFI=0
def mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll,estado_scroll, sensibilidad,mano):
   # Mouse Move

    global  FFD,FFI
    sw, sh = pyautogui.size()
    y = int((mouse_Y - 240) * sensibilidad * sh / 480)

    if mano == 0:
        x = int((mouse_X - 320) * sensibilidad * sw / 640) 
    else:
        x = int(mouse_X * sensibilidad * sw / 640)

    try: pyautogui.moveTo(x, y, _pause=False)
    except: pass

    pyautogui.scroll(100*scroll)

    if estado_scroll == 0:
        FFD = f_tecla(FFD, BD, 'right')
        FFI = f_tecla(FFI, BI, 'left')


estado_scroll = 0
valor_anterior_y = 0
def mouse_gesto_operacion_scroll(hl):
    
    global estado_scroll, valor_anterior_y
    scroll = 0

    gesto_estado_subir = (abs(hl.landmark[8].x - hl.landmark[13].x)*640 < 30) and estado_scroll == 0
    gesto_estado_bajar = (abs(hl.landmark[5].y - hl.landmark[8].y)*480 < 50) and estado_scroll == 0
    gesto_estado_reposo = (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x)

    # ESTADOS
    if(gesto_estado_subir): estado_scroll = 1
    if(gesto_estado_bajar): estado_scroll = 2
    if(gesto_estado_reposo): estado_scroll = 0


    # SALIDAS
    #SALIDA SUBIDA
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        #print("1")
        scroll = 1

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5  and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0


    #SALIDA BAJADA    
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y              
        scroll = 0

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y                
        scroll = -1
        
    return scroll,estado_scroll


def frecuencia_muestreo():
    global tiempo_previo, Fs
    t_actual = time.time()
    dt = t_actual - tiempo_previo
    tiempo_previo = t_actual
    if dt > 0: 
        tiempos_fps.append(dt)
        
    if len(tiempos_fps) > 50: 
        tiempos_fps.pop(0)

    if len(tiempos_fps) == 50:
        Fs = 1 / (sum(tiempos_fps) / 50)
        actualizar_coeficientes()
    #print(f"FPS: {Fs:.2f}", end="\r")
    
def procesar_loop():
    frecuencia_muestreo()
    
    ret, frame = cap.read()
    if not ret: 
        return

    coord_mano = Deteccion_mano_openCV(frame)

    if coord_mano.multi_hand_landmarks:
        hl = coord_mano.multi_hand_landmarks[0] # hl: hand landmark (puntos de referencia)
        
        BI,BD,mouse_X,mouse_Y,scroll,estado_scroll = Deteccion_gestos(hl)      
        mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll, estado_scroll, sensibilidad,mano)
     

# ==============================
# CONTROL DE UI Y RECURSOS
# ==============================
def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): 
        cap.release()
        
    boton_inicio.setText("Iniciar")

def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    if not ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            return QMessageBox.critical(None, "Error", "Cámara no disponible")
        ejecutando = True
        boton_inicio.setText("Pausar")
        timer.start(30)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()

def cerrar_aplicacion(event):
    apagar_recursos()
    hands.close()
    cv2.destroyAllWindows()
    event.accept()

# ==============================
# SETUP DE PYQT5 (SIN CLASE)
# ==============================
# EVITA EL ERROR DE "QApplication instance already exists"
app = QApplication.instance() or QApplication(sys.argv)

ventana = QWidget()
ventana.setWindowTitle("Control Gestual")
ventana.setFixedSize(500, 450)
ventana.closeEvent = cerrar_aplicacion # Inyectamos el cierre

layout = QVBoxLayout()
lbl_logo = QLabel()
pix = QPixmap("mouse virtual logo.jpg")
if not pix.isNull(): lbl_logo.setPixmap(pix.scaled(200, 150, Qt.KeepAspectRatio))
lbl_logo.setAlignment(Qt.AlignCenter)

boton_inicio = QPushButton("Iniciar")
boton_inicio.clicked.connect(iniciar_programa)

# Mano
radio_der = QRadioButton("Derecha")
radio_der.setChecked(True)
radio_der.toggled.connect(lambda: globals().update(mano=0 if radio_der.isChecked() else 1))

radio_izq = QRadioButton("Izquierda")

# Sliders
sld_sens = QSlider(Qt.Horizontal); sld_sens.setRange(1, 10)
sld_sens.setValue(sensibilidad)
sld_sens.valueChanged.connect(lambda v: globals().update(sensibilidad=v))

sld_fc = QSlider(Qt.Horizontal)
sld_fc.setRange(1, 10)
sld_fc.setValue(fc)
sld_fc.valueChanged.connect(lambda v: (globals().update(fc=v), actualizar_coeficientes()))

layout.addWidget(lbl_logo)
layout.addWidget(boton_inicio)

layout.addWidget(QLabel("Mano"))
layout.addWidget(radio_der)
layout.addWidget(radio_izq)

layout.addWidget(QLabel("Sensibilidad"))
layout.addWidget(sld_sens)

layout.addWidget(QLabel("Filtro (FC)"))
layout.addWidget(sld_fc)

ventana.setLayout(layout)
actualizar_coeficientes()
timer.timeout.connect(procesar_loop)

ventana.show()
app.exec_()

0